# 🚨 Emergency Response System — Harnessing Deep Learning to Optimize Emergency Response
**IAU Graduation Project | ARTI 521**

Complete unified pipeline: EfficientNet-B0 baseline → YOLOv11 classifier → TimeSformer (Path 1) → YOLO11+ByteTrack (Path 2) → Qwen2.5-VL Triage

In [ ]:
# ⚠️ RESTART RUNTIME AFTER THIS CELL
!pip install -q torchmetrics
!pip install -q transformers accelerate
!pip install -q decord
!pip install -q scikit-learn
!pip install -q ultralytics
!pip install -q lapx
!pip install -q bitsandbytes
!pip install -q qwen-vl-utils
!pip install -q albumentations
!pip install -q pyyaml
print("✅ All dependencies installed.")
print("⚠️  Restart runtime now, then skip this cell and run from Cell 2.")

In [ ]:
# ============================================================
# Mount Drive + All Imports + Unified Path Constants
# ============================================================

# ── Drive ──────────────────────────────────────────────
from google.colab import drive
import os
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
else:
    print("✅ Drive already mounted")

# ── Standard library ───────────────────────────────────
import json, shutil, random, time, math, re, textwrap, gc, pickle, glob
from pathlib import Path
from collections import defaultdict, deque, Counter

# ── Numeric / vision ─────────────────────────────────
import numpy as np
import cv2
import yaml
from PIL import Image

# ── PyTorch ──────────────────────────────────────────
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.cuda.amp import GradScaler, autocast

# ── TorchMetrics ───────────────────────────────────
from torchmetrics import Accuracy, Precision, Recall, F1Score, AUROC
from torchmetrics import ConfusionMatrix

# ── HuggingFace ──────────────────────────────────
from transformers import (AutoImageProcessor, TimesformerForVideoClassification,
                          Qwen2_5_VLForConditionalGeneration, AutoProcessor,
                          BitsAndBytesConfig)
try:
    from qwen_vl_utils import process_vision_info
    QWEN_VL_UTILS = True
except ImportError:
    QWEN_VL_UTILS = False
    print("⚠️  qwen-vl-utils not found — VLM section requires it")

# ── Ultralytics ──────────────────────────────────
from ultralytics import YOLO

# ── Decord (fast video decoder) ──────────────────────
try:
    import decord
    decord.bridge.set_bridge('torch')
    from decord import VideoReader, cpu, gpu
    USE_DECORD = True
    print("Decord available — using fast video decoder")
except ImportError:
    USE_DECORD = False
    print("Decord not found — falling back to cv2")

# ── Visualization ──────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as patches
from tqdm.notebook import tqdm
from IPython.display import display, Image as IPImage

# ── sklearn ─────────────────────────────────────────
from sklearn.metrics import confusion_matrix, classification_report, roc_curve
import seaborn as sns

# ── Reproducibility ─────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# ── Unified Path Constants ───────────────────────────
CADP_ROOT       = Path('/content/drive/MyDrive/Car_accident_detection')
CADP_DATASET    = CADP_ROOT / 'Dataset'
FRAMES_DIR      = CADP_DATASET / 'manual' / 'extracted_frames'
VIDEOS_DIR      = CADP_DATASET / 'forth_investigation'
ANNO_FILE       = CADP_DATASET / 'manual' / 'annotations_1531762138.1303267.json'

WORK_DIR        = Path('/content/drive/MyDrive/Colab Notebooks/grad-project/program-memory')
CACHE_DIR       = WORK_DIR / 'cache'
CHECKPOINT_DIR  = WORK_DIR / 'checkpoints'
RUNS_DIR        = WORK_DIR / 'runs'
DATASET_DIR     = WORK_DIR / 'cls_dataset_v2'
RESULTS_DIR     = WORK_DIR / 'triage_results'
TRACKING_DIR    = WORK_DIR / 'tracking_results'

for d in [CACHE_DIR, CHECKPOINT_DIR, RUNS_DIR, DATASET_DIR, RESULTS_DIR, TRACKING_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Device ─────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    print(f"🖥️  GPU: {torch.cuda.get_device_name(0)} "
          f"({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)")
else:
    print("❌ No GPU — Go to Runtime > Change runtime type > T4 GPU")

# ── Sanity checks ──────────────────────────────────
for label, path in [("extracted_frames", FRAMES_DIR),
                    ("forth_investigation", VIDEOS_DIR),
                    ("annotations json",   ANNO_FILE)]:
    exists = path.exists()
    print(f"  {'✅' if exists else '❌'}  {label}: {path}")

print("\n✅ All imports and paths ready")

## Section 1 — Dataset Building

In [ ]:
# ============================================================
# 2.1 Explore Clip Structure
# ============================================================
if clip_folders:
    print(f"📂 {len(clip_folders)} clip folders in extracted_frames/\n")

    # Show first 5 clips
    for folder in clip_folders[:5]:
        frames = sorted(folder.glob('*.jpg'))
        print(f"  📁 {folder.name}/ → {len(frames)} frames", end="")
        if frames:
            print(f"  (first: {frames[0].name}, last: {frames[-1].name})")
        else:
            print()

    if len(clip_folders) > 5:
        print(f"  ... and {len(clip_folders) - 5} more clips")

    # Stats — cached to avoid 8-minute Drive scan on every run
    cache_file = Path("/content/drive/MyDrive/grad-project/program-memory/cache/frame_counts.json")

    if cache_file.exists():
        with open(cache_file) as f:
            all_frame_counts = json.load(f)
        print("✅ Frame counts loaded from cache")
    else:
        print("⏳ Counting frames (runs once, then cached)...")
        all_frame_counts = [len(list(f.glob('*.jpg'))) for f in clip_folders]
        cache_file.parent.mkdir(parents=True, exist_ok=True)
        with open(cache_file, 'w') as f:
            json.dump(all_frame_counts, f)
        print("✅ Frame counts saved to cache")

    print(f"\n📊 Frame count stats:")
    print(f"   Min: {min(all_frame_counts)}, Max: {max(all_frame_counts)}, "
          f"Avg: {np.mean(all_frame_counts):.0f}, Total: {sum(all_frame_counts)}")

else:
    print("❌ No clip folders found")

# ============================================================
# 2.2 Visualize Sample Frames from a Clip
# ============================================================
if clip_folders:
    sample_clip = clip_folders[0]
    frames = sorted(sample_clip.glob('*.jpg'))

    print(f"📂 Clip: {sample_clip.name} ({len(frames)} frames)\n")

    # Show start (normal), middle (ambiguous zone), and end (accident)
    indices = [0, min(50, len(frames)-1), min(90, len(frames)-1),
               min(105, len(frames)-1), len(frames)-1]
    labels = ["Frame 0\n(Normal)", "Frame 50\n(Normal)", "Frame 90\n(Normal boundary)",
              "Frame 105\n(Accident start)", f"Frame {len(frames)-1}\n(Late accident)"]

    fig, axes = plt.subplots(1, min(5, len(indices)), figsize=(20, 4))
    for ax, idx, label in zip(axes, indices, labels):
        if idx < len(frames):
            img = cv2.imread(str(frames[idx]))
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            ax.imshow(img)
            ax.set_title(label, fontsize=10)
        ax.axis('off')

    plt.suptitle(f"Clip {sample_clip.name} — Frame Timeline", fontsize=14)
    plt.tight_layout()
    plt.show()

# ============================================================
# 2.3 Explore Annotation File
# ============================================================
if ANNO_FILE.exists():
    with open(ANNO_FILE, 'r') as f:
        raw_annotations = json.load(f)
    print(f"✅ Annotation file loaded: {len(raw_annotations)} entries")
    print(f"   Keys are YouTube video IDs (e.g., {list(raw_annotations.keys())[:3]})")
    print(f"\n   Note: These map to videos in forth_investigation/, not extracted_frames/")
    print(f"   For extracted_frames/ we use the CADP paper convention:")
    print(f"   frames 0-90 = normal, 91-104 = skip, 105+ = accident")
else:
    print("⚠️  No annotation file found (expected for extracted_frames/ workflow)")
    print("   Using CADP paper convention: frames 0-90 = normal, 105+ = accident")

In [ ]:
# ============================================================
# 3.1  Build Classification Dataset  (RESUMABLE)
# ============================================================
#
# POSITIVES  — extracted_frames/{clip}/
#   • Skip clips with < ACCIDENT_START frames (9.3% of clips)
#   • From remaining frames >= ACCIDENT_START, sample evenly
#     up to MAX_ACC_PER_CLIP frames per clip
#   • Frames named: 0.jpg, 1.jpg, 2.jpg ...
#
# NEGATIVES  — forth_investigation/{video}.mp4
#   • Slide a 2-second window over full video duration
#   • Exclude any window that overlaps an accident annotation
#     (+ 1-second safety buffer each side)
#   • From safe windows, sample up to MAX_NORM_PER_VIDEO
#   • Extract the middle frame of each safe window via OpenCV
#
# Both sources are on Google Drive → local copy to /tmp
# before frame reads to eliminate per-frame Drive latency.
# ============================================================

import math

ACCIDENT_START    = 105       # frame index where accident begins
FPS               = 30.0      # confirmed 30fps for all clips
WINDOW_SEC        = 2.0       # length of each normal-traffic window
BUFFER_SEC        = 1.0       # exclusion buffer around accident zones
MAX_ACC_PER_CLIP  = 5         # max accident frames sampled per clip
MAX_NORM_PER_VID  = 30        # max normal frames sampled per video
TRAIN_RATIO       = 0.70
VAL_RATIO         = 0.20

PROGRESS_FILE  = CACHE_DIR / 'build_progress_v2.json'
CACHE_DONE     = CACHE_DIR / 'dataset_v2_complete.json'
LOCAL_TMP      = Path('/content/tmp_work')

# ── helpers ────────────────────────────────────────────────
def load_progress():
    if PROGRESS_FILE.exists():
        with open(PROGRESS_FILE) as f:
            return json.load(f)
    return {
        'done_clips': [],
        'done_videos': [],
        'counts': {'train':{'acc':0,'norm':0},
                   'val':  {'acc':0,'norm':0},
                   'test': {'acc':0,'norm':0}},
        'skipped': {'short_clip':0, 'corrupt':0, 'no_safe_window':0}
    }

def save_progress(p):
    with open(PROGRESS_FILE, 'w') as f:
        json.dump(p, f)

def dataset_done():
    return CACHE_DONE.exists()

def safe_imread(path):
    """Read image, return None on any failure."""
    try:
        img = cv2.imread(str(path))
        return img if (img is not None and img.size > 0) else None
    except Exception:
        return None

def sample_evenly(items, n):
    """Sample up to n items evenly spread across the list."""
    if len(items) <= n:
        return items
    step = len(items) / n
    return [items[int(i * step)] for i in range(n)]

# ── early exit ─────────────────────────────────────────────
if dataset_done():
    with open(CACHE_DONE) as f:
        done_info = json.load(f)
    print("⚡ Dataset already built! Skipping.\n")
    for split in ['train', 'val', 'test']:
        for label in ['accident', 'no_accident']:
            n = len(list((DATASET_DIR/split/label).glob('*.jpg')))
            print(f"   {split}/{label}: {n:,} frames")

else:
    # ── create output folders ───────────────────────────────
    for split in ['train', 'val', 'test']:
        for label in ['accident', 'no_accident']:
            (DATASET_DIR / split / label).mkdir(parents=True, exist_ok=True)

    progress = load_progress()
    done_clips  = set(progress['done_clips'])
    done_videos = set(progress['done_videos'])
    counts      = progress['counts']
    skipped     = progress['skipped']

    # ── deterministic split assignment ─────────────────────
    all_clips  = sorted([d for d in FRAMES_DIR.iterdir() if d.is_dir()])
    all_videos = sorted([v for v in VIDEOS_DIR.glob('*.mp4')])

    random.Random(SEED).shuffle(all_clips)
    random.Random(SEED).shuffle(all_videos)

    n_train = int(len(all_clips) * TRAIN_RATIO)
    n_val   = int(len(all_clips) * VAL_RATIO)
    clip_split = {}
    for i, c in enumerate(all_clips):
        clip_split[c.name] = ('train' if i < n_train
                              else 'val' if i < n_train + n_val
                              else 'test')

    n_train_v = int(len(all_videos) * TRAIN_RATIO)
    n_val_v   = int(len(all_videos) * VAL_RATIO)
    vid_split = {}
    for i, v in enumerate(all_videos):
        vid_split[v.name] = ('train' if i < n_train_v
                             else 'val' if i < n_train_v + n_val_v
                             else 'test')

    remaining_clips  = [c for c in all_clips  if c.name not in done_clips]
    remaining_videos = [v for v in all_videos if v.name not in done_videos]

    total_work = len(remaining_clips) + len(remaining_videos)
    print(f"🔄 Resuming: {len(done_clips)} clips done, "
          f"{len(done_videos)} videos done")
    print(f"   Remaining: {len(remaining_clips)} clips + "
          f"{len(remaining_videos)} videos\n")

    # ══════════════════════════════════════════════════════
    # PHASE 1 — POSITIVES from extracted_frames
    # ══════════════════════════════════════════════════════
    print("── Phase 1: extracting POSITIVE frames ──")
    for clip_dir in tqdm(remaining_clips, desc="Accident clips"):
        frames_all = sorted(clip_dir.glob('*.jpg'),
                            key=lambda p: int(p.stem))
        total_frames = len(frames_all)

        if total_frames < ACCIDENT_START:
            skipped['short_clip'] += 1
            progress['done_clips'].append(clip_dir.name)
            save_progress(progress)
            continue

        # frames in accident zone
        acc_frames = [f for f in frames_all
                      if int(f.stem) >= ACCIDENT_START]

        chosen = sample_evenly(acc_frames, MAX_ACC_PER_CLIP)
        split  = clip_split[clip_dir.name]

        # copy clip to local tmp (eliminates per-frame Drive reads)
        if LOCAL_TMP.exists(): shutil.rmtree(LOCAL_TMP)
        LOCAL_TMP.mkdir(parents=True)
        for src in chosen:
            shutil.copy2(src, LOCAL_TMP / src.name)

        saved_count = 0
        for src_name in [f.name for f in chosen]:
            local_path = LOCAL_TMP / src_name
            img = safe_imread(local_path)
            if img is None:
                skipped['corrupt'] += 1
                continue
            dst = (DATASET_DIR / split / 'accident'
                   / f"{clip_dir.name}_f{src_name.replace('.jpg',''):>06}.jpg")
            shutil.copy2(str(local_path), str(dst))
            counts[split]['acc'] += 1
            saved_count += 1

        progress['done_clips'].append(clip_dir.name)
        progress['counts']  = counts
        progress['skipped'] = skipped
        save_progress(progress)

    if LOCAL_TMP.exists(): shutil.rmtree(LOCAL_TMP)

    # ══════════════════════════════════════════════════════
    # PHASE 2 — NEGATIVES from forth_investigation
    # ══════════════════════════════════════════════════════
    print("\n── Phase 2: extracting NEGATIVE frames ──")
    for video_path in tqdm(remaining_videos, desc="Normal videos"):
        vid_name   = video_path.name
        acc_windows = get_accident_windows(vid_name)

        cap  = cv2.VideoCapture(str(video_path))
        fps  = cap.get(cv2.CAP_PROP_FPS) or FPS
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        total_dur = total_frames / fps

        # generate all 2-second safe windows
        safe_windows = []
        t = 0.0
        while t + WINDOW_SEC <= total_dur:
            if window_is_safe(t, t + WINDOW_SEC, acc_windows, BUFFER_SEC):
                safe_windows.append(t)
            t += WINDOW_SEC

        if not safe_windows:
            skipped['no_safe_window'] += 1
            cap.release()
            progress['done_videos'].append(vid_name)
            save_progress(progress)
            continue

        chosen_starts = sample_evenly(safe_windows, MAX_NORM_PER_VID)
        split = vid_split[vid_name]

        for idx, t_start in enumerate(chosen_starts):
            t_mid      = t_start + WINDOW_SEC / 2.0
            frame_idx  = int(t_mid * fps)
            cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
            ret, frame = cap.read()
            if not ret or frame is None or frame.size == 0:
                skipped['corrupt'] += 1
                continue
            vid_stem = vid_name.replace('.mp4', '').replace('-','_')
            dst = (DATASET_DIR / split / 'no_accident'
                   / f"{vid_stem}_{idx:04d}.jpg")
            cv2.imwrite(str(dst), frame)
            counts[split]['norm'] += 1

        cap.release()
        progress['done_videos'].append(vid_name)
        progress['counts']  = counts
        progress['skipped'] = skipped
        save_progress(progress)

    # ── finalize ───────────────────────────────────────────
    with open(CACHE_DONE, 'w') as f:
        json.dump({'counts': counts, 'skipped': skipped}, f, indent=2)
    PROGRESS_FILE.unlink(missing_ok=True)

    print("\n✅ Dataset complete!\n")
    for split in ['train', 'val', 'test']:
        a = counts[split]['acc']
        n = counts[split]['norm']
        print(f"   {split:5s}: {a+n:,} images  "
              f"(accident={a:,}, normal={n:,})")
    print(f"\n   Skipped — short clips:    {skipped['short_clip']}")
    print(f"   Skipped — corrupt frames: {skipped['corrupt']}")
    print(f"   Skipped — no safe window: {skipped['no_safe_window']}")

In [ ]:
# ============================================================
# 3.2  Visualize dataset samples + class balance
# ============================================================
fig, axes = plt.subplots(2, 4, figsize=(20, 9))
fig.suptitle('Dataset samples (train split)', fontsize=15)

for row, label in enumerate(['accident', 'no_accident']):
    samples = list((DATASET_DIR / 'train' / label).glob('*.jpg'))
    if not samples:
        print(f"⚠️  No {label} samples found yet — run cell 3.1 first")
        continue
    chosen = random.sample(samples, min(4, len(samples)))
    for col, p in enumerate(chosen):
        img = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)
        axes[row][col].imshow(img)
        color = 'red' if label == 'accident' else 'steelblue'
        axes[row][col].set_title(label, color=color, fontsize=10)
        axes[row][col].axis('off')

plt.tight_layout()
plt.show()

print("\n📊 Class balance:")
for split in ['train', 'val', 'test']:
    acc  = len(list((DATASET_DIR/split/'accident').glob('*.jpg')))
    norm = len(list((DATASET_DIR/split/'no_accident').glob('*.jpg')))
    tot  = acc + norm
    if tot:
        print(f"   {split:5s}: {tot:,} total | "
              f"accident {acc/tot*100:.0f}% | normal {norm/tot*100:.0f}%")

In [ ]:
# 2.1 FPS-aware helpers — time-based sampling (FIX #2)
def get_clip_fps(clip_dir: Path) -> float:
    """
    Estimate FPS from clip folder by reading the parent video if possible.
    Falls back to 30fps if unavailable.
    """
    # Try matching clip folder name to a video file
    clip_name = clip_dir.name
    vid_matches = list(VIDEOS_DIR.glob(f'{clip_name}*.mp4')) + \
                  list(VIDEOS_DIR.glob(f'*{clip_name}*.mp4'))
    if vid_matches:
        cap = cv2.VideoCapture(str(vid_matches[0]))
        fps = cap.get(cv2.CAP_PROP_FPS)
        cap.release()
        if fps and fps > 0:
            return fps
    return 30.0


def sample_positive_frames(frame_paths: list, fps: float, n_frames: int) -> list:
    """
    Time-based sampling — FIX for FPS fairness.

    CADP accident starts at ~3.5s (frame 105 at 30fps).
    We sample a CLIP_SEC window CENTERED on the accident time.
    This means the model always sees the same real-world duration
    regardless of whether the video is 15fps or 60fps.

    Window: [accident_sec - 0.5s, accident_sec + 1.5s]
    → model sees the moment just before + the full aftermath
    """
    total = len(frame_paths)
    if total == 0:
        return []

    # Convert CADP accident frame to real seconds
    # Use min() to handle clips with non-standard FPS
    accident_frame_idx = min(int(ACCIDENT_SEC * fps), total - 1)

    # Define time window: 0.5s before → 1.5s after accident
    window_start_f = max(0, accident_frame_idx - int(0.5 * fps))
    window_end_f   = min(total - 1, accident_frame_idx + int(1.5 * fps))

    if window_end_f <= window_start_f:
        # Very short clip — use full clip
        window_start_f, window_end_f = 0, total - 1

    # Sample n_frames uniformly from the window
    # This guarantees accident frames appear
    window_len = window_end_f - window_start_f
    if window_len < n_frames:
        indices = sorted(random.choices(
            range(window_start_f, window_end_f + 1), k=n_frames))
    else:
        stride  = window_len / n_frames
        indices = [window_start_f + int(i * stride) for i in range(n_frames)]
        # Jitter slightly to avoid overfitting on exact indices
        jitter  = max(1, int(stride * 0.2))
        indices = [max(window_start_f,
                       min(window_end_f, idx + random.randint(-jitter, jitter)))
                   for idx in indices]

    return [frame_paths[i] for i in sorted(set(indices))[:n_frames]]


def sample_negative_frames(frame_paths: list, fps: float, n_frames: int) -> list:
    """Sample n_frames uniformly by TIME from a normal clip."""
    total = len(frame_paths)
    if total == 0:
        return []
    # Avoid last 0.5s (might have partial accident in boundary clips)
    safe_end = max(n_frames, total - int(0.5 * fps))
    stride   = safe_end / n_frames
    indices  = [min(total - 1, int(i * stride)) for i in range(n_frames)]
    return [frame_paths[i] for i in indices]


print("Time-based samplers ready")
print(f"  Accident window : [{ACCIDENT_SEC-0.5:.1f}s, {ACCIDENT_SEC+1.5:.1f}s] around accident")
print(f"  Frames sampled  : {NUM_FRAMES} per clip")
print(f"  Window duration : {CLIP_SEC}s (same real-world time for all FPS)")

# 2.2 Build / load dataset splits (FPS-aware)
CACHE_FILE = CACHE_DIR / 'videovit_splits_v2.json'

if CACHE_FILE.exists():
    with open(CACHE_FILE) as f:
        split_data = json.load(f)
    print(f"Loaded cached splits: {CACHE_FILE.name}")
else:
    print("Building time-based splits...")
    positive_samples = []
    fps_cache = {}

    for clip_dir in tqdm(sorted(FRAMES_DIR.iterdir()), desc="Positive"):
        if not clip_dir.is_dir(): continue
        fps = fps_cache.get(clip_dir.name) or get_clip_fps(clip_dir)
        fps_cache[clip_dir.name] = fps
        frame_paths = sorted(clip_dir.glob('*.jpg'),
                              key=lambda p: int(p.stem) if p.stem.isdigit() else 0)
        # Clip must have at least accident_sec * fps frames
        if len(frame_paths) < int(ACCIDENT_SEC * fps) - 5:
            continue
        sampled = sample_positive_frames(frame_paths, fps, NUM_FRAMES)
        if len(sampled) == NUM_FRAMES:
            positive_samples.append(([str(p) for p in sampled], 1))

    with open(ANNOT_JSON) as f:
        annotations = json.load(f)
    from collections import defaultdict
    exclude = defaultdict(list)
    for ann in annotations.get('annotations', []):
        vid = ann.get('video_id','')
        ts, te = ann.get('t_start',0), ann.get('t_end',5)
        exclude[vid].append((ts-1.0, te+1.0))

    negative_samples = []
    for vf in tqdm(sorted(VIDEOS_DIR.glob('*.mp4')), desc="Negative"):
        cap   = cv2.VideoCapture(str(vf))
        fps_v = cap.get(cv2.CAP_PROP_FPS) or 30.0
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        cap.release()
        if total < NUM_FRAMES * 10: continue
        excl = exclude.get(vf.stem, [])
        dur  = total / fps_v
        count = 0; attempts = 0
        while count < 20 and attempts < 100:
            attempts += 1
            ss = random.uniform(0, max(0, dur - CLIP_SEC - 0.5))
            es = ss + CLIP_SEC
            if any(not(es < a or ss > b) for a,b in excl): continue
            sf, ef = int(ss*fps_v), min(total-1, int(es*fps_v))
            cl = ef - sf
            if cl < NUM_FRAMES: continue
            stride = cl / NUM_FRAMES
            idxs = [sf + int(i*stride) for i in range(NUM_FRAMES)]
            negative_samples.append(({'video':str(vf),'indices':idxs,'fps':fps_v}, 0))
            count += 1

    random.shuffle(positive_samples); random.shuffle(negative_samples)
    def spl(lst):
        n=len(lst); return lst[:int(n*.70)], lst[int(n*.70):int(n*.85)], lst[int(n*.85):]
    ptr,pva,pte = spl(positive_samples)
    ntr,nva,nte = spl(negative_samples)
    split_data  = {'train':ptr+ntr,'val':pva+nva,'test':pte+nte}
    random.shuffle(split_data['train'])
    with open(CACHE_FILE,'w') as f: json.dump(split_data,f)
    print(f"Cached to {CACHE_FILE.name}")

for s,d in split_data.items():
    na = sum(1 for x in d if (x[1] if isinstance(x,list) else x[-1])==1)
    print(f"  {s}: {len(d)} clips  ({na} accident / {len(d)-na} normal)")

In [ ]:
# ============================================================
# 3.3 Create data.yaml
# ============================================================

# For YOLO classification, data.yaml just points to the root folder
# with train/val/test subdirectories containing class folders
DATA_YAML_PATH = DATASET_DIR / 'data.yaml'

data_yaml = {
    'path': str(DATASET_DIR),
    'train': 'train',
    'val': 'val',
    'test': 'test',
    'nc': 2,
    'names': ['no_accident', 'accident']
}

with open(DATA_YAML_PATH, 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False, sort_keys=False)

print(f"✅ data.yaml created at: {DATA_YAML_PATH}")
print()
with open(DATA_YAML_PATH) as f:
    print(f.read())

## Section 2 — EfficientNet-B0 Baseline

In [ ]:
# ============================================================
# 4.1  PyTorch Dataset + Transforms
# ============================================================

class AccidentDataset(Dataset):
    def __init__(self, root: Path, split: str, transform=None):
        self.items     = []
        self.transform = transform
        self.class_to_idx = {'accident': 1, 'no_accident': 0}
        for label, idx in self.class_to_idx.items():
            folder = root / split / label
            if folder.exists():
                for p in folder.glob('*.jpg'):
                    self.items.append((p, idx))
        random.Random(SEED).shuffle(self.items)

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        path, label = self.items[idx]
        img = cv2.imread(str(path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        if self.transform:
            img = self.transform(img)
        return img, torch.tensor(label, dtype=torch.long)

# ImageNet normalization stats
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

train_transform = T.Compose([
    T.ToPILImage(),
    T.Resize((256, 256)),
    T.RandomCrop(224),
    T.RandomHorizontalFlip(),
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])

eval_transform = T.Compose([
    T.ToPILImage(),
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])

train_ds = AccidentDataset(DATASET_DIR, 'train', train_transform)
val_ds   = AccidentDataset(DATASET_DIR, 'val',   eval_transform)
test_ds  = AccidentDataset(DATASET_DIR, 'test',  eval_transform)

train_dl = DataLoader(train_ds, batch_size=32, shuffle=True,
                      num_workers=2, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=32, shuffle=False,
                      num_workers=2, pin_memory=True)
test_dl  = DataLoader(test_ds,  batch_size=32, shuffle=False,
                      num_workers=2, pin_memory=True)

print(f"✅ Datasets ready")
print(f"   Train : {len(train_ds):,} samples  ({len(train_dl)} batches)")
print(f"   Val   : {len(val_ds):,}  samples  ({len(val_dl)} batches)")
print(f"   Test  : {len(test_ds):,}  samples  ({len(test_dl)} batches)")

# ============================================================
# 4.2  EfficientNet-B0 — partial freeze + stronger regularization
# ============================================================
# Changes from v1:
#   • Freeze first 5 of 8 feature blocks  →  trainable params
#     drop from 4.0M to ~1.2M  →  fixes overfitting
#   • Dropout 0.3 → 0.5                   →  more regularization
#   • weight_decay 1e-4 → 1e-3            →  more regularization
# ============================================================

effnet_model = models.efficientnet_b0(
    weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1
)

# ── Partial freeze: lock first 5 feature blocks ────────────
# EfficientNet-B0 has features[0..8]:
#   0 = stem conv
#   1..8 = MBConv blocks (8 blocks total)
# We freeze blocks 0-5, fine-tune blocks 6-8 + classifier head
FREEZE_UP_TO = 6   # freeze features[0] through features[5]

for i, block in enumerate(effnet_model.features):
    if i < FREEZE_UP_TO:
        for param in block.parameters():
            param.requires_grad = False

# ── Replace classifier head ────────────────────────────────
in_features = effnet_model.classifier[1].in_features
effnet_model.classifier = nn.Sequential(
    nn.Dropout(p=0.5, inplace=True),   # was 0.3
    nn.Linear(in_features, 2)
)
effnet_model = effnet_model.to(DEVICE)

# ── Class weights ──────────────────────────────────────────
n_acc  = len(list((DATASET_DIR/'train'/'accident').glob('*.jpg')))
n_norm = len(list((DATASET_DIR/'train'/'no_accident').glob('*.jpg')))
n_total = n_acc + n_norm
w_acc  = n_total / (2 * n_acc)  if n_acc  > 0 else 1.0
w_norm = n_total / (2 * n_norm) if n_norm > 0 else 1.0
class_weights = torch.tensor([w_norm, w_acc], dtype=torch.float).to(DEVICE)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = AdamW(
    filter(lambda p: p.requires_grad, effnet_model.parameters()),
    lr=1e-4,
    weight_decay=1e-3      # was 1e-4
)
scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)

total_params = sum(p.numel() for p in effnet_model.parameters())
train_params = sum(p.numel() for p in effnet_model.parameters() if p.requires_grad)
frozen_params = total_params - train_params

print(f"✅ EfficientNet-B0 (partial freeze) ready")
print(f"   Total params     : {total_params/1e6:.1f}M")
print(f"   Frozen params    : {frozen_params/1e6:.1f}M  (features[0-5])")
print(f"   Trainable params : {train_params/1e6:.2f}M  (features[6-8] + head)")
print(f"   Dropout          : 0.5")
print(f"   Weight decay     : 1e-3")
print(f"   Class weights    : accident={w_acc:.3f}, normal={w_norm:.3f}")
print(f"   Device           : {DEVICE}")


In [ ]:
# ============================================================
# 5.1  Training — v2 (partial freeze, stronger regularization)
# ============================================================
# Saves to efficientnet_b0_v2_best.pt — v1 model is preserved.
# ============================================================

NUM_EPOCHS = 60
PATIENCE   = 15    # slightly more patience

BEST_PT   = CHECKPOINT_DIR / 'efficientnet_b0_v2_best.pt'
LAST_PT   = CHECKPOINT_DIR / 'efficientnet_b0_v2_last.pt'
HISTORY_F = CACHE_DIR       / 'train_history_v2.json'

def make_metrics():
    return {
        'acc'  : Accuracy(task='binary').to(DEVICE),
        'prec' : Precision(task='binary').to(DEVICE),
        'rec'  : Recall(task='binary').to(DEVICE),
        'f1'   : F1Score(task='binary').to(DEVICE),
        'auc'  : AUROC(task='binary').to(DEVICE),
    }

def compute_metrics(m):
    return {k: v.compute().item() for k, v in m.items()}

def run_epoch(loader, train=True):
    effnet_model.train(train)
    metrics    = make_metrics()
    total_loss = 0.0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            with torch.amp.autocast('cuda'):
                logits = effnet_model(imgs)
                loss   = criterion(logits, labels)
            if train:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
            total_loss += loss.item()
            probs = torch.softmax(logits, dim=1)[:, 1]
            preds = logits.argmax(dim=1)
            metrics['acc'].update(preds, labels)
            metrics['prec'].update(preds, labels)
            metrics['rec'].update(preds, labels)
            metrics['f1'].update(preds, labels)
            metrics['auc'].update(probs, labels)
    return total_loss / len(loader), compute_metrics(metrics)

scaler = torch.amp.GradScaler('cuda')

# ── Resume support ─────────────────────────────────────────
start_epoch = 0
best_val_f1 = 0.0
no_improve  = 0
history     = {'train': [], 'val': []}

if LAST_PT.exists():
    ckpt = torch.load(LAST_PT, map_location=DEVICE)
    effnet_model.load_state_dict(ckpt['model'])
    optimizer.load_state_dict(ckpt['optimizer'])
    scheduler.load_state_dict(ckpt['scheduler'])
    start_epoch = ckpt['epoch'] + 1
    best_val_f1 = ckpt.get('best_val_f1', 0.0)
    no_improve  = ckpt.get('no_improve', 0)
    if HISTORY_F.exists():
        with open(HISTORY_F) as f:
            history = json.load(f)
    print(f"🔄 Resumed from epoch {start_epoch}  (best F1={best_val_f1:.4f})")
else:
    print("🚀 Starting fresh training (v2 — partial freeze)")

for epoch in range(start_epoch, NUM_EPOCHS):
    t0 = time.time()
    train_loss, train_m = run_epoch(train_dl, train=True)
    val_loss,   val_m   = run_epoch(val_dl,   train=False)
    scheduler.step()

    history['train'].append({'loss': train_loss, **train_m})
    history['val'].append(  {'loss': val_loss,   **val_m})
    with open(HISTORY_F, 'w') as f:
        json.dump(history, f)

    elapsed = time.time() - t0
    print(f"Ep {epoch+1:3d}/{NUM_EPOCHS} | "
          f"loss {train_loss:.4f}/{val_loss:.4f} | "
          f"F1 {train_m['f1']:.3f}/{val_m['f1']:.3f} | "
          f"AUC {val_m['auc']:.3f} | "
          f"Prec {val_m['prec']:.3f} Rec {val_m['rec']:.3f} | "
          f"{elapsed:.0f}s")

    ckpt = {'epoch': epoch, 'model': effnet_model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'scheduler': scheduler.state_dict(),
            'best_val_f1': best_val_f1, 'no_improve': no_improve}
    torch.save(ckpt, LAST_PT)

    if val_m['f1'] > best_val_f1:
        best_val_f1 = val_m['f1']
        no_improve  = 0
        torch.save(effnet_model.state_dict(), BEST_PT)
        print(f"   💾 New best F1={best_val_f1:.4f} — saved to {BEST_PT.name}")
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f"\n⏹️  Early stopping ({PATIENCE} epochs no improvement)")
            break

print(f"\n✅ Training complete. Best val F1 = {best_val_f1:.4f}")
print(f"   v1 best: {CHECKPOINT_DIR}/efficientnet_b0_best.pt")
print(f"   v2 best: {BEST_PT}")


In [ ]:
# ============================================================
# 5.2  Training curves
# ============================================================
if not HISTORY_F.exists():
    print("No training history found — run training first.")
else:
    with open(HISTORY_F) as f:
        history = json.load(f)

    epochs = range(1, len(history['train']) + 1)
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle('Training History', fontsize=15)

    metrics_to_plot = ['loss', 'f1', 'auc', 'acc', 'prec', 'rec']
    titles = ['Loss', 'F1 Score', 'AUC-ROC',
              'Accuracy', 'Precision', 'Recall']

    for ax, key, title in zip(axes.flatten(), metrics_to_plot, titles):
        tr = [e[key] for e in history['train']]
        vl = [e[key] for e in history['val']]
        ax.plot(epochs, tr, label='train', color='steelblue')
        ax.plot(epochs, vl, label='val',   color='coral')
        ax.set_title(title)
        ax.set_xlabel('Epoch')
        ax.legend()
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

# ============================================================
# 6.1  Full evaluation on test set
# ============================================================

# Load best weights
if BEST_PT.exists():
    effnet_model.load_state_dict(torch.load(BEST_PT, map_location=DEVICE))
    print(f"✅ Loaded best weights: {BEST_PT.name}")
else:
    print("⚠️  best.pt not found — using current model weights")

effnet_model.eval()
all_preds, all_labels, all_probs = [], [], []

with torch.no_grad():
    for imgs, labels in tqdm(test_dl, desc="Evaluating"):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        logits = effnet_model(imgs)
        probs  = torch.softmax(logits, dim=1)[:, 1]
        preds  = logits.argmax(dim=1)
        all_preds.append(preds.cpu())
        all_labels.append(labels.cpu())
        all_probs.append(probs.cpu())

all_preds  = torch.cat(all_preds)
all_labels = torch.cat(all_labels)
all_probs  = torch.cat(all_probs)

# Compute metrics
test_metrics = {}
for name, metric in [
    ('Accuracy',  Accuracy(task='binary')),
    ('Precision', Precision(task='binary')),
    ('Recall',    Recall(task='binary')),
    ('F1',        F1Score(task='binary')),
    ('AUC-ROC',   AUROC(task='binary')),
]:
    if name == 'AUC-ROC':
        test_metrics[name] = metric(all_probs, all_labels).item()
    else:
        test_metrics[name] = metric(all_preds, all_labels).item()

print("\n" + "="*50)
print("📊 TEST SET RESULTS")
print("="*50)
for k, v in test_metrics.items():
    print(f"  {k:12s}: {v:.4f}")
print("="*50)

# Confusion matrix
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
cm = confusion_matrix(all_labels.numpy(), all_preds.numpy())
disp = ConfusionMatrixDisplay(cm, display_labels=['normal', 'accident'])
fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Confusion Matrix — Test Set', fontsize=13)
plt.tight_layout()
plt.show()

# ROC curve
from sklearn.metrics import roc_curve
fpr, tpr, _ = roc_curve(all_labels.numpy(), all_probs.numpy())
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr, tpr, color='coral', lw=2,
        label=f"AUC = {test_metrics['AUC-ROC']:.3f}")
ax.plot([0,1],[0,1],'--', color='gray')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve — Test Set')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# ============================================================
# 6.2  Sample predictions on test images
# ============================================================
effnet_model.eval()
samples_acc  = list((DATASET_DIR/'test'/'accident').glob('*.jpg'))
samples_norm = list((DATASET_DIR/'test'/'no_accident').glob('*.jpg'))
samples = (random.sample(samples_acc,  min(4, len(samples_acc))) +
           random.sample(samples_norm, min(4, len(samples_norm))))
random.shuffle(samples)

cols = 4
rows = math.ceil(len(samples) / cols)
fig, axes = plt.subplots(rows, cols, figsize=(5*cols, 4*rows))
axes = np.array(axes).flatten()
fig.suptitle('Test set predictions', fontsize=14)

for ax, p in zip(axes, samples):
    gt = 1 if p.parent.name == 'accident' else 0
    img_bgr = cv2.imread(str(p))
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    t = eval_transform(img_rgb).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        prob = torch.softmax(effnet_model(t), dim=1)[0, 1].item()
    pred = 1 if prob >= 0.5 else 0
    correct = pred == gt

    ax.imshow(img_rgb)
    color  = 'green' if correct else 'red'
    symbol = '✅' if correct else '❌'
    gt_name   = 'accident' if gt   == 1 else 'normal'
    pred_name = 'accident' if pred == 1 else 'normal'
    ax.set_title(f"{symbol} GT:{gt_name}\nPred:{pred_name} ({prob:.0%})",
                 color=color, fontsize=9)
    ax.axis('off')

for ax in axes[len(samples):]:
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 7.1  Inference with temporal smoothing
# ============================================================
# Added: 5-frame sliding window majority vote.
# A detection fires only when >= VOTE_THRESHOLD of the last
# VOTE_WINDOW frames say "accident" — eliminates single spikes.
# ============================================================

VOTE_WINDOW    = 5
VOTE_THRESHOLD = 3
CONF_THRESHOLD = 0.5

def infer_video(effnet_model, video_path, sample_every_n=15, max_sec=120):
    video_path = Path(video_path)
    if not video_path.exists():
        print(f"❌ Not found: {video_path}")
        return [], []

    cap        = cv2.VideoCapture(str(video_path))
    fps        = cap.get(cv2.CAP_PROP_FPS) or 30
    total      = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    max_frames = min(total, int(max_sec * fps))

    print(f"🎬 {video_path.name}: {total} frames @ {fps:.1f}fps")
    effnet_model.eval()

    raw_timeline  = []
    smooth_alerts = []
    recent_preds  = []

    for i in tqdm(range(max_frames), desc="Inferring", leave=False):
        ret, frame = cap.read()
        if not ret: break
        if i % sample_every_n != 0: continue

        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        t = eval_transform(frame_rgb).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            prob = torch.softmax(effnet_model(t), dim=1)[0, 1].item()

        raw_pred = 1 if prob >= CONF_THRESHOLD else 0
        raw_timeline.append({'frame': i, 'time_sec': i/fps,
                              'prob': prob, 'raw': raw_pred})

        recent_preds.append(raw_pred)
        if len(recent_preds) > VOTE_WINDOW:
            recent_preds.pop(0)
        smooth = 1 if sum(recent_preds) >= VOTE_THRESHOLD else 0
        smooth_alerts.append({'frame': i, 'time_sec': i/fps,
                               'prob': prob, 'smooth': smooth})

    cap.release()

    raw_acc    = [t for t in raw_timeline  if t['raw']    == 1]
    smooth_acc = [t for t in smooth_alerts if t['smooth'] == 1]

    print(f"\n   Raw detections  : {len(raw_acc)}/{len(raw_timeline)} frames")
    print(f"   Smoothed alerts : {len(smooth_acc)}/{len(smooth_alerts)} frames")

    if smooth_acc:
        first = smooth_acc[0]
        peak  = max(raw_timeline, key=lambda x: x['prob'])
        print(f"   First alert at  : {first['time_sec']:.1f}s")
        print(f"   Peak confidence : {peak['prob']:.0%} at {peak['time_sec']:.1f}s")
    else:
        print("   No sustained accident detected")

    return raw_timeline, smooth_alerts

def plot_timeline(raw_tl, smooth_tl, video_name=""):
    times_r  = [t['time_sec'] for t in raw_tl]
    probs    = [t['prob']     for t in raw_tl]
    raw_d    = [t['raw']      for t in raw_tl]
    times_s  = [t['time_sec'] for t in smooth_tl]
    smooth_d = [t['smooth']   for t in smooth_tl]

    fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)
    fig.suptitle(f'Detection timeline — {video_name}', fontsize=13)

    axes[0].fill_between(times_r, raw_d, alpha=0.35, color='red')
    axes[0].plot(times_r, raw_d, 'r-', lw=1.5)
    axes[0].set_ylabel('Raw (per-frame)')
    axes[0].axhline(0.5, color='gray', ls='--', lw=0.8)

    axes[1].fill_between(times_s, smooth_d, alpha=0.4, color='darkorange')
    axes[1].plot(times_s, smooth_d, color='darkorange', lw=1.5)
    axes[1].set_ylabel(f'Smoothed ({VOTE_THRESHOLD}/{VOTE_WINDOW} vote)')
    axes[1].axhline(0.5, color='gray', ls='--', lw=0.8)

    axes[2].plot(times_r, probs, 'b-', lw=1.2)
    axes[2].axhline(CONF_THRESHOLD, color='gray', ls='--',
                    lw=0.8, label=f'threshold={CONF_THRESHOLD}')
    axes[2].set_ylabel('Accident probability')
    axes[2].set_xlabel('Time (seconds)')
    axes[2].legend()

    for ax in axes: ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

# ── Load best model (v2 if exists, else v1) ────────────────
best_v2 = CHECKPOINT_DIR / 'efficientnet_b0_v2_best.pt'
best_v1 = CHECKPOINT_DIR / 'efficientnet_b0_best.pt'
best_to_load = best_v2 if best_v2.exists() else best_v1
effnet_model.load_state_dict(torch.load(best_to_load, map_location=DEVICE))
print(f"✅ Loaded: {best_to_load.name}\n")

test_vids = sorted(VIDEOS_DIR.glob('*.mp4'))
if test_vids:
    raw_tl, smooth_tl = infer_video(effnet_model, test_vids[0])
    if raw_tl:
        plot_timeline(raw_tl, smooth_tl, test_vids[0].name)
else:
    print("No videos found in forth_investigation/")


In [ ]:
# ============================================================
# 8.1  Export to ONNX
# ============================================================

# Install required packages (takes ~30 seconds, run once)
import subprocess
subprocess.run(["pip", "install", "-q", "onnx", "onnxscript"], check=True)

import onnx

best_v2 = CHECKPOINT_DIR / 'efficientnet_b0_v2_best.pt'
best_v1 = CHECKPOINT_DIR / 'efficientnet_b0_best.pt'
best_to_load = best_v2 if best_v2.exists() else best_v1

if best_to_load.exists():
    effnet_model.load_state_dict(torch.load(best_to_load, map_location=DEVICE))
    effnet_model.eval()

    dummy    = torch.randn(1, 3, 224, 224).to(DEVICE)
    onnx_tmp = '/content/efficientnet_b0_accident.onnx'

    torch.onnx.export(
        effnet_model, dummy, onnx_tmp,
        input_names=['image'],
        output_names=['logits'],
        dynamic_axes={'image': {0: 'batch'}},
        opset_version=12,
        do_constant_folding=True,
        dynamo=False,          # force legacy TorchScript exporter
    )

    label     = 'v2' if best_v2.exists() else 'v1'
    onnx_dest = CHECKPOINT_DIR / f'efficientnet_b0_{label}.onnx'
    shutil.copy2(onnx_tmp, onnx_dest)
    size_mb = onnx_dest.stat().st_size / 1e6
    print(f"✅ ONNX exported : {onnx_dest.name}  ({size_mb:.1f} MB)")
    print(f"   Opset version : 12")
    print(f"   Input shape   : [batch, 3, 224, 224]")
    print(f"   Output shape  : [batch, 2]  — apply softmax(dim=1)[:,1] for accident prob")
else:
    print("❌ No checkpoint found — run training first.")

# ============================================================
# 8.2  Final summary
# ============================================================
counts = {}
for split in ['train','val','test']:
    counts[split] = {}
    for label in ['accident','no_accident']:
        counts[split][label] = len(
            list((DATASET_DIR/split/label).glob('*.jpg')))

total = sum(v for s in counts.values() for v in s.values())
print("=" * 60)
print("📋  EXPERIMENT SUMMARY")
print("=" * 60)
print(f"  Model       : EfficientNet-B0 (torchvision IMAGENET1K_V1)")
print(f"  Task        : Binary classification (accident / normal)")
print(f"  Input size  : 224 × 224 RGB")
print(f"  Total imgs  : {total:,}")
for split in ['train','val','test']:
    a = counts[split]['accident']
    n = counts[split]['no_accident']
    print(f"  {split:5s}       : {a+n:,}  (acc={a:,}  norm={n:,})")
print()
print("  📂 Drive paths:")
print(f"     Dataset    : {DATASET_DIR}")
print(f"     Best model : {BEST_PT}")
print(f"     ONNX       : {CHECKPOINT_DIR/'efficientnet_b0_accident.onnx'}")
print(f"     History    : {HISTORY_F}")
print("=" * 60)

## Section 3 — YOLOv11 Classifier

In [ ]:
# ============================================================
# 4.1 Training Configuration
# ============================================================

MODEL_VARIANT = 'yolo11s-cls.pt'   # Classification model (small)

TRAINING_CONFIG = {
    'data':       str(DATASET_DIR),
    'epochs':     100,
    'batch':      32,            # Classification can use larger batch
    'imgsz':      224,           # Standard classification size (faster than 640)
    'patience':   20,            # Early stopping
    'save_period': 5,            # Checkpoint every 5 epochs
    'device':     0,
    'workers':    2,
    'project':    str(RUNS_DIR),
    'name':       'accident_cls_yolov11',
    'exist_ok':   True,
    'pretrained':  True,
    'optimizer':  'AdamW',
    'lr0':        0.001,
    'lrf':        0.01,
    'cos_lr':     True,
    'warmup_epochs': 3,
    'seed':       SEED,

    # Augmentation (conservative for CCTV footage)
    'hsv_h': 0.015,
    'hsv_s': 0.5,
    'hsv_v': 0.3,
    'degrees': 3.0,        # Slight rotation only
    'translate': 0.1,
    'scale': 0.3,
    'flipud': 0.0,         # No vertical flip for CCTV
    'fliplr': 0.5,
    'erasing': 0.1,
}

print("✅ Training configuration ready")
print(f"   Model:      {MODEL_VARIANT}")
print(f"   Epochs:     {TRAINING_CONFIG['epochs']} (patience={TRAINING_CONFIG['patience']})")
print(f"   Batch:      {TRAINING_CONFIG['batch']}")
print(f"   Image size: {TRAINING_CONFIG['imgsz']}")
print(f"   Output:     {RUNS_DIR / 'accident_cls_yolov11'}")

# ============================================================
# 4.2 Train (with Auto-Resume Support)
# ============================================================
# Set RESUME = True if training was interrupted by a disconnect.
# The model will pick up from the last saved checkpoint.
# ============================================================

RESUME = False  # ← Set True after a disconnect

RUN_DIR = RUNS_DIR / 'accident_cls_yolov11'

if RESUME:
    last_pt = RUN_DIR / 'weights' / 'last.pt'
    if last_pt.exists():
        print(f"🔄 Resuming from: {last_pt}")
        yolo_cls_model = YOLO(str(last_pt))
        results = yolo_cls_model.train(resume=True)
    else:
        print("⚠️  No last.pt found — starting fresh")
        RESUME = False

if not RESUME:
    print(f"🚀 Starting fresh training with {MODEL_VARIANT}\n")
    yolo_cls_model = YOLO(MODEL_VARIANT)
    results = yolo_cls_model.train(**TRAINING_CONFIG)

# Backup best weights to checkpoint dir
best_pt = RUN_DIR / 'weights' / 'best.pt'
if best_pt.exists():
    backup_path = CHECKPOINT_DIR / 'accident_cls_best.pt'
    shutil.copy2(str(best_pt), str(backup_path))
    print(f"\n💾 Best weights backed up to: {backup_path}")

print("\n✅ Training complete!")

In [ ]:
# ============================================================
# 5.1 Validate on Test Set
# ============================================================

# Load best yolo_cls_model
best_pt = CHECKPOINT_DIR / 'accident_cls_best.pt'
if not best_pt.exists():
    best_pt = RUN_DIR / 'weights' / 'best.pt'

if best_pt.exists():
    eval_model = YOLO(str(best_pt))
    print(f"✅ Loaded best yolo_cls_model: {best_pt.name}")
else:
    eval_model = yolo_cls_model
    print("⚠️  Using last trained yolo_cls_model (best.pt not found)")

# Run validation on test split
test_results = eval_model.val(
    data=str(DATASET_DIR),
    split='test',
    batch=32,
    imgsz=224,
    verbose=True
)

print("\n" + "=" * 50)
print("📊 TEST SET RESULTS")
print("=" * 50)
print(f"  Top-1 Accuracy: {test_results.top1:.4f}")
print(f"  Top-5 Accuracy: {test_results.top5:.4f}")
print("=" * 50)

# ============================================================
# 5.2 Display Training Curves
# ============================================================

RUN_DIR = RUNS_DIR / 'accident_cls_yolov11'

plot_files = ['results.png', 'confusion_matrix.png', 'confusion_matrix_normalized.png']

for plot_name in plot_files:
    plot_path = RUN_DIR / plot_name
    if plot_path.exists():
        print(f"\n📈 {plot_name}")
        display(IPImage(filename=str(plot_path), width=800))
    else:
        print(f"⚠️  {plot_name} not found")

# ============================================================
# 5.3 Inference on Sample Test Images
# ============================================================

test_accident = list((DATASET_DIR / 'test' / 'accident').glob('*.jpg'))
test_normal   = list((DATASET_DIR / 'test' / 'no_accident').glob('*.jpg'))

# Mix accident and normal samples
samples = []
if test_accident:
    samples += random.sample(test_accident, min(4, len(test_accident)))
if test_normal:
    samples += random.sample(test_normal, min(4, len(test_normal)))
random.shuffle(samples)

n_show = min(8, len(samples))
if n_show == 0:
    print("❌ No test images found")
else:
    cols = min(4, n_show)
    rows = (n_show + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(5*cols, 5*rows))
    if rows == 1 and cols == 1:
        axes = np.array([axes])
    axes = axes.flatten()

    fig.suptitle('Test Set Predictions', fontsize=16, fontweight='bold')

    for idx in range(len(axes)):
        ax = axes[idx]
        if idx < n_show:
            img_path = samples[idx]

            # Get ground truth from folder name
            gt_label = img_path.parent.name

            # Run prediction
            pred = eval_model.predict(str(img_path), verbose=False)
            pred_class = pred[0].probs.top1
            pred_name  = pred[0].names[pred_class]
            pred_conf  = pred[0].probs.top1conf.item()

            correct = (pred_name == gt_label)

            img = cv2.imread(str(img_path))
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            ax.imshow(img)

            color = 'green' if correct else 'red'
            symbol = '✅' if correct else '❌'
            ax.set_title(f"{symbol} GT: {gt_label}\nPred: {pred_name} ({pred_conf:.0%})",
                        fontsize=9, color=color)
        ax.axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================
# 6.1 Run Inference on a Video
# ============================================================

def classify_video(model, video_path, conf_threshold=0.5, max_frames=500, sample_every=3):
    """
    Run frame-by-frame classification on a video.
    Returns timeline of predictions.
    """
    video_path = Path(video_path)
    if not video_path.exists():
        print(f"❌ Video not found: {video_path}")
        return None

    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    print(f"🎬 {video_path.name}: {total} frames @ {fps:.0f}fps ({total/fps:.1f}s)")

    timeline = []
    frame_idx = 0

    for _ in tqdm(range(min(total, max_frames)), desc="Classifying"):
        ret, frame = cap.read()
        if not ret:
            break

        if frame_idx % sample_every == 0:
            # Save temp frame for prediction
            temp_path = '/content/temp_frame.jpg'
            cv2.imwrite(temp_path, frame)

            pred = model.predict(temp_path, verbose=False)
            pred_class = pred[0].probs.top1
            pred_name  = pred[0].names[pred_class]
            pred_conf  = pred[0].probs.top1conf.item()

            timeline.append({
                'frame': frame_idx,
                'time_sec': frame_idx / fps,
                'prediction': pred_name,
                'confidence': pred_conf
            })

        frame_idx += 1

    cap.release()

    # Find first accident detection
    accidents = [t for t in timeline if t['prediction'] == 'accident' and t['confidence'] >= conf_threshold]

    print(f"\n✅ Processed {frame_idx} frames")
    print(f"   Accident detections: {len(accidents)} / {len(timeline)} sampled frames")
    if accidents:
        first = accidents[0]
        peak  = max(accidents, key=lambda x: x['confidence'])
        print(f"   First accident at: {first['time_sec']:.1f}s (conf: {first['confidence']:.0%})")
        print(f"   Peak confidence:   {peak['confidence']:.0%} at {peak['time_sec']:.1f}s")

    return timeline

# --- Run on a sample video from CADP ---
# Check forth_investigation for videos
video_dir = CADP_DATASET / 'forth_investigation'
if video_dir.exists():
    test_videos = sorted(video_dir.glob('*.mp4'))[:1]
    if test_videos:
        timeline = classify_video(eval_model, test_videos[0])

        # Plot timeline
        if timeline:
            times = [t['time_sec'] for t in timeline]
            is_acc = [1 if t['prediction'] == 'accident' else 0 for t in timeline]
            confs  = [t['confidence'] for t in timeline]

            fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
            ax1.plot(times, is_acc, 'r-', linewidth=1.5)
            ax1.fill_between(times, is_acc, alpha=0.3, color='red')
            ax1.set_ylabel('Accident Detected')
            ax1.set_title(f'Accident Detection Timeline — {test_videos[0].name}')

            ax2.plot(times, confs, 'b-', linewidth=1)
            ax2.set_ylabel('Confidence')
            ax2.set_xlabel('Time (seconds)')

            plt.tight_layout()
            plt.show()
    else:
        print("No .mp4 files found in forth_investigation/")
else:
    print("forth_investigation/ not found — upload a video to test")

# ============================================================
# 7.1 Export Model to ONNX
# ============================================================

best_pt = CHECKPOINT_DIR / 'accident_cls_best.pt'
if not best_pt.exists():
    best_pt = RUN_DIR / 'weights' / 'best.pt'

if best_pt.exists():
    export_model = YOLO(str(best_pt))

    onnx_path = export_model.export(format='onnx', imgsz=224, simplify=True)
    print(f"\n✅ ONNX exported: {onnx_path}")

    # Backup to Drive
    if Path(onnx_path).exists():
        dest = CHECKPOINT_DIR / 'accident_cls_best.onnx'
        shutil.copy2(onnx_path, str(dest))
        print(f"💾 Backed up to: {dest}")
else:
    print("❌ No best weights found. Train the model first (Section 4).")

# ============================================================
# 7.2 Summary Report
# ============================================================

# Count dataset
counts = {}
for split in ['train', 'val', 'test']:
    counts[split] = {}
    for label in ['accident', 'no_accident']:
        counts[split][label] = len(list((DATASET_DIR / split / label).glob('*.jpg')))

total_images = sum(c for s in counts.values() for c in s.values())

print("=" * 60)
print("📋 TRAINING SUMMARY")
print("=" * 60)
print()
print(f"  Project:        Accident Detection (YOLOv11)")
print(f"  Task:           Classification (accident vs no_accident)")
print(f"  Model:          {MODEL_VARIANT}")
print(f"  Dataset:        CADP — manual/extracted_frames/")
print(f"  Total Images:   {total_images}")
print()
for split in ['train', 'val', 'test']:
    acc  = counts[split]['accident']
    norm = counts[split]['no_accident']
    print(f"  {split:>5}: {acc + norm} images (accident={acc}, normal={norm})")
print()
print(f"  📂 Files on Drive:")
print(f"     Dataset:     {DATASET_DIR}")
print(f"     Best model:  {CHECKPOINT_DIR / 'accident_cls_best.pt'}")
print(f"     ONNX:        {CHECKPOINT_DIR / 'accident_cls_best.onnx'}")
print(f"     Runs:        {RUNS_DIR}")
print()
print("=" * 60)
print()
print("🎓 Next Steps:")
print("  1. Review confusion matrix — check for systematic errors")
print("  2. Try larger model (yolo11m-cls.pt) if accuracy is low")
print("  3. Add more data from outsource/ and Lijun/ folders")
print("  4. Try imgsz=640 for better small-object handling")
print("  5. Integrate into your Flask/FastAPI backend")

## Section 4 — Path 1: TimeSformer (Video ViT)

In [ ]:
# 3.2 Load TimeSformer — unfreeze last 4 blocks (FIX #3 — use T4 properly)
MODEL_ID = "facebook/timesformer-base-finetuned-k400"
print(f"Loading {MODEL_ID}...")

processor = AutoImageProcessor.from_pretrained(MODEL_ID)
timesformer_model = TimesformerForVideoClassification.from_pretrained(
    MODEL_ID,
    num_labels=2,
    ignore_mismatched_sizes=True,

)

# Unfreeze last 4 transformer blocks + classifier head
# TimeSformer-base has 12 blocks (0-11)
UNFREEZE_BLOCKS = 4
for name, param in timesformer_model.named_parameters():
    param.requires_grad = False   # freeze everything first

for name, param in timesformer_model.named_parameters():
    # Unfreeze last UNFREEZE_BLOCKS blocks
    for block_idx in range(12 - UNFREEZE_BLOCKS, 12):
        if f'timesformer.encoder.layer.{block_idx}.' in name:
            param.requires_grad = True
    # Always unfreeze classifier head
    if 'classifier' in name:
        param.requires_grad = True

# Replace head for binary classification
timesformer_model.classifier = nn.Linear(timesformer_model.classifier.in_features, 2)
timesformer_model = timesformer_model.to(DEVICE)

total  = sum(p.numel() for p in timesformer_model.parameters())
train  = sum(p.numel() for p in timesformer_model.parameters() if p.requires_grad)
print(f"Total params     : {total/1e6:.1f}M")
print(f"Trainable params : {train/1e6:.1f}M  (last {UNFREEZE_BLOCKS} blocks + head)")
print(f"VRAM used        : {torch.cuda.memory_allocated()/1e9:.2f} GB")

# Build DataLoaders — optimized for T4 (FIX #1 continued)
BATCH = 16   # 4× old batch size

train_ds = FastVideoDataset(split_data['train'], processor)
val_ds   = FastVideoDataset(split_data['val'],   processor)
test_ds  = FastVideoDataset(split_data['test'],  processor)

# num_workers=4, prefetch_factor=4, persistent_workers=True
# These are the key CPU bottleneck fixes
DL_KWARGS = dict(batch_size=BATCH, num_workers=4, pin_memory=True,
                 prefetch_factor=4, persistent_workers=True)

train_dl = DataLoader(train_ds, shuffle=True,  **DL_KWARGS)
val_dl   = DataLoader(val_ds,   shuffle=False, **DL_KWARGS)
test_dl  = DataLoader(test_ds,  shuffle=False, **DL_KWARGS)

print(f"Batch size : {BATCH}  |  Workers : 4  |  Prefetch : 4")
print(f"Train : {len(train_dl)} batches  |  Val : {len(val_dl)}")

In [ ]:
# 3.1 Fast Dataset — Decord video decoder (FIX #1)
class FastVideoDataset(torch.utils.data.Dataset):
    """
    Uses Decord for video clips (3-5× faster than cv2).
    Falls back to cv2 if Decord unavailable.
    All video indices are pre-computed — no FPS logic in __getitem__.
    """
    def __init__(self, samples, processor):
        self.samples = samples
        self.proc    = processor

    def __len__(self): return len(self.samples)

    def _read_frames_from_paths(self, paths):
        frames = []
        for p in paths:
            img = cv2.imread(str(p))
            if img is not None:
                frames.append(Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)))
            else:
                frames.append(Image.fromarray(np.zeros((224,224,3), np.uint8)))
        return frames

    def _read_frames_from_video(self, info):
        vid   = info['video']
        idxs  = info['indices']
        if USE_DECORD:
            try:
                vr = VideoReader(vid, ctx=cpu(0), num_threads=2)
                # Clamp indices to valid range
                idxs = [min(idx, len(vr)-1) for idx in idxs]
                frames_tensor = vr.get_batch(idxs)  # [T, H, W, C] torch tensor
                frames = [Image.fromarray(frames_tensor[i].numpy()) for i in range(len(idxs))]
                return frames
            except Exception:
                pass  # fall through to cv2
        # cv2 fallback
        cap    = cv2.VideoCapture(vid); frames = []
        for idx in idxs:
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ret, fr = cap.read()
            frames.append(Image.fromarray(cv2.cvtColor(fr, cv2.COLOR_BGR2RGB))
                          if ret else Image.fromarray(np.zeros((224,224,3), np.uint8)))
        cap.release(); return frames

    def __getitem__(self, i):
        sample, label = self.samples[i]
        frames = (self._read_frames_from_paths(sample)
                  if isinstance(sample, list)
                  else self._read_frames_from_video(sample))
        # Ensure exactly NUM_FRAMES
        while len(frames) < NUM_FRAMES:
            frames.append(frames[-1] if frames else
                          Image.fromarray(np.zeros((224,224,3), np.uint8)))
        frames = frames[:NUM_FRAMES]
        pv = self.proc(images=frames, return_tensors="pt")['pixel_values'].squeeze(0)
        return pv, torch.tensor(label, dtype=torch.long)


print("FastVideoDataset ready (Decord:", USE_DECORD, ")")

In [ ]:
# 4.1 Training with AMP (FIX #1 + #3)
# AMP = Automatic Mixed Precision
# - Forward pass in fp16 (fast, uses less VRAM)
# - Loss scaled to prevent fp16 underflow
# - ~2× throughput on T4 vs fp32

import torch.nn as nn

EPOCHS=30; PATIENCE=10
BEST_PT = CKPT_DIR/'timesformer_v2_best.pt'
LAST_PT = CKPT_DIR/'timesformer_v2_last.pt'
HIST_F  = CACHE_DIR/'timesformer_v2_history.json'

na    = sum(1 for x in split_data['train'] if (x[1] if isinstance(x,list) else x[-1])==1)
n_neg = len(split_data['train']) - na
n_tot = na + n_neg
cw    = torch.tensor([n_tot/(2*n_neg), n_tot/(2*na)], dtype=torch.float16).to(DEVICE)
crit  = nn.CrossEntropyLoss(weight=cw)

opt = AdamW(filter(lambda p:p.requires_grad,timesformer_model.parameters()),
            lr=2e-5, weight_decay=1e-3)
sch = CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=1e-7)

# GradScaler for AMP — prevents fp16 underflow
scaler = torch.amp.GradScaler('cuda')

def make_M():
    return {k:v.to(DEVICE) for k,v in {
        'f1':F1Score(task='binary'),'acc':Accuracy(task='binary'),
        'auc':AUROC(task='binary')}.items()}

def run(loader, train=True):
    timesformer_model.train(train); M=make_M(); ls=0.0; n=0
    ctx = torch.enable_grad() if train else torch.no_grad()
    pbar = tqdm(loader, desc='train' if train else 'val ',
                leave=False, dynamic_ncols=True)
    with ctx:
        for clips, labels in pbar:
            clips, labels = clips.to(DEVICE), labels.to(DEVICE)
            if train: opt.zero_grad(set_to_none=True)
            # AMP forward pass
            with torch.amp.autocast('cuda'):
                logits = timesformer_model(pixel_values=clips).logits
                loss   = crit(logits.float(), labels)
            if train:
                scaler.scale(loss).backward()
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(timesformer_model.parameters(), 1.0)
                scaler.step(opt); scaler.update()
            ls += loss.item(); n += 1
            probs = torch.softmax(logits.float(),dim=1)[:,1]
            preds = logits.argmax(dim=1)
            M['f1'].update(preds,labels); M['acc'].update(preds,labels)
            M['auc'].update(probs,labels)
            if train:
                pbar.set_postfix({'loss':f'{ls/n:.3f}',
                                   'f1':f'{M["f1"].compute():.3f}',
                                   'vram':f'{torch.cuda.memory_allocated()/1e9:.1f}GB'})
    return ls/n, {k:v.compute().item() for k,v in M.items()}

se=0; bf=0.0; ni=0; hist={'train':[],'val':[]}
if LAST_PT.exists():
    ck=torch.load(LAST_PT,map_location=DEVICE,weights_only=False)
    timesformer_model.load_state_dict(ck['model']); opt.load_state_dict(ck['opt'])
    sch.load_state_dict(ck['sch']); scaler.load_state_dict(ck['scaler'])
    se=ck['epoch']+1; bf=ck.get('bf',0); ni=ck.get('ni',0)
    if HIST_F.exists():
        with open(HIST_F) as f: hist=json.load(f)
    print(f"Resumed ep {se}, best F1={bf:.4f}")
else:
    print("Starting training (fp16 AMP, batch=16, 26M trainable params)")

for ep in range(se, EPOCHS):
    t0=time.time()
    tl,tm=run(train_dl,True); vl,vm=run(val_dl,False); sch.step()
    hist['train'].append({'loss':tl,**tm}); hist['val'].append({'loss':vl,**vm})
    with open(HIST_F,'w') as f: json.dump(hist,f)
    vram=torch.cuda.memory_allocated()/1e9
    print(f"Ep {ep+1:3d}/{EPOCHS} | loss {tl:.3f}/{vl:.3f} | "
          f"F1 {tm['f1']:.3f}/{vm['f1']:.3f} | AUC {vm['auc']:.3f} | "
          f"VRAM {vram:.1f}GB | {time.time()-t0:.0f}s")
    ck={'epoch':ep,'model':timesformer_model.state_dict(),'opt':opt.state_dict(),
        'sch':sch.state_dict(),'scaler':scaler.state_dict(),'bf':bf,'ni':ni}
    torch.save(ck, LAST_PT)
    if vm['f1']>bf: bf=vm['f1']; ni=0; torch.save(timesformer_model.state_dict(),BEST_PT); print(f"  Best F1={bf:.4f}")
    else:
        ni+=1
        if ni>=PATIENCE: print("Early stop"); break
print(f"Best val F1={bf:.4f}")

In [ ]:
# 4.2 Test evaluation + accident bbox (FIX #4)
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

timesformer_model.load_state_dict(torch.load(BEST_PT, map_location=DEVICE, weights_only=True))
timesformer_model.eval()
pa,la,pra=[],[],[]
with torch.no_grad():
    for clips,labels in tqdm(test_dl,desc="Testing"):
        with autocast():
            logits=timesformer_model(pixel_values=clips.to(DEVICE)).logits
        probs=torch.softmax(logits.float(),dim=1)[:,1]
        pa.extend(logits.argmax(1).cpu().tolist()); la.extend(labels.tolist())
        pra.extend(probs.cpu().tolist())

print(classification_report(la,pa,target_names=['normal','accident']))
cm=confusion_matrix(la,pa); tn,fp,fn,tp=cm.ravel()
print(f"TN={tn} FP={fp} FN={fn} TP={tp}  FPR={fp/(fp+tn)*100:.1f}%")

fig,ax=plt.subplots(1,2,figsize=(12,4))
sns.heatmap(cm,annot=True,fmt='d',ax=ax[0],
            xticklabels=['normal','accident'],yticklabels=['normal','accident'])
ax[0].set_title('Confusion Matrix — TimeSformer v2 (AMP, 16 batch)')
ax[1].hist([pra[i] for i,l in enumerate(la) if l==0],bins=30,alpha=0.6,label='normal',color='blue')
ax[1].hist([pra[i] for i,l in enumerate(la) if l==1],bins=30,alpha=0.6,label='accident',color='red')
ax[1].axvline(0.5,color='k',ls='--'); ax[1].legend()
ax[1].set_title('Score distribution')
plt.tight_layout(); plt.show()

In [ ]:
# 5.1 Video inference + accident bbox for VLM (FIX #4)
# TimeSformer is a classifier. The bbox comes from YOLO run on the
# trigger frame — giving VLM a 20%-padded crop of the accident zone.

def infer_video_with_bbox(video_path, window_sec=2.0, stride_sec=0.5,
                           thresh=0.6, yolo_model=None):
    """
    Sliding window inference.
    On detection: run YOLO on trigger frame to get accident bbox.
    Returns timeline + first accident event with bbox.
    """
    from ultralytics import YOLO as _YOLO
    if yolo_model is None:
        try:    yolo_local = _YOLO('yolo11n.pt')
        except: yolo_local = _YOLO('yolov8n.pt')
        yolo_local.to(DEVICE)
    else:
        yolo_local = yolo_model

    VEHICLE_IDS = {2,3,5,7,1}
    def get_bbox_from_frame(frame_bgr):
        """Run YOLO and return accident zone with 20% padding (FIX #4)."""
        h,w=frame_bgr.shape[:2]
        r=yolo_local.predict(frame_bgr,verbose=False,conf=0.3,classes=list(VEHICLE_IDS))[0]
        bboxes=[]
        if r.boxes is not None and len(r.boxes)>0:
            for box in r.boxes:
                x1,y1,x2,y2=[int(v) for v in box.xyxy[0].tolist()]
                bboxes.append([x1,y1,x2,y2])
        if not bboxes: return None
        # Union of all detected vehicle boxes
        ax1=min(b[0] for b in bboxes); ay1=min(b[1] for b in bboxes)
        ax2=max(b[2] for b in bboxes); ay2=max(b[3] for b in bboxes)
        # 20% padding (FIX #4)
        pad_x=int((ax2-ax1)*0.20); pad_y=int((ay2-ay1)*0.20)
        return [max(0,ax1-pad_x),max(0,ay1-pad_y),
                min(w,ax2+pad_x),min(h,ay2+pad_y)]

    cap   = cv2.VideoCapture(str(video_path))
    fps   = cap.get(cv2.CAP_PROP_FPS) or 30
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    wf    = int(window_sec * fps)
    sf_s  = int(stride_sec * fps)
    print(f"  {Path(video_path).name}: {total/fps:.1f}s @ {fps:.0f}fps")
    timeline=[]; timesformer_model.eval(); first_event=None

    for start in range(0, total-wf, sf_s):
        idxs=[start+int(i*wf/NUM_FRAMES) for i in range(NUM_FRAMES)]
        frames=[]
        for idx in idxs:
            cap.set(cv2.CAP_PROP_POS_FRAMES,idx); ret,fr=cap.read()
            frames.append(Image.fromarray(cv2.cvtColor(fr,cv2.COLOR_BGR2RGB))
                          if ret else Image.fromarray(np.zeros((224,224,3),np.uint8)))
        pv=processor(images=frames,return_tensors="pt")['pixel_values'].to(DEVICE)
        with torch.no_grad(), autocast():
            prob=torch.softmax(timesformer_model(pixel_values=pv).logits.float(),dim=1)[0,1].item()
        det = prob >= thresh
        timeline.append({'t':start/fps,'prob':prob,'det':det})
        if det and first_event is None:
            # Get the middle frame of this window as trigger frame
            mid_idx = idxs[NUM_FRAMES//2]
            cap.set(cv2.CAP_PROP_POS_FRAMES,mid_idx); ret,trigger_fr=cap.read()
            bbox = get_bbox_from_frame(trigger_fr) if ret else None
            first_event={'time_sec':start/fps,'prob':prob,
                          'collision_bbox':bbox,'trigger_frame':trigger_fr if ret else None}

    cap.release()
    alerts=[t for t in timeline if t['det']]
    print(f"  Windows: {len(timeline)} | Alerts: {len(alerts)}")
    if first_event:
        print(f"  First alert: {first_event['time_sec']:.1f}s  bbox={first_event['collision_bbox']}")
    return timeline, first_event

vids=sorted(VIDEOS_DIR.glob('*.mp4'))
if vids:
    tl, ev = infer_video_with_bbox(vids[0])
    fig,axes=plt.subplots(1,2 if ev and ev.get('trigger_frame') is not None else 1,
                           figsize=(14 if ev else 8, 4))
    ax_tl = axes[0] if isinstance(axes,(list,np.ndarray)) else axes
    ax_tl.fill_between([t['t'] for t in tl],[0.5*t['det'] for t in tl],
                        alpha=0.3,color='red',label='detected')
    ax_tl.plot([t['t'] for t in tl],[t['prob'] for t in tl],'b-',lw=1.5)
    ax_tl.axhline(0.6,color='gray',ls='--',lw=0.8); ax_tl.set_xlabel('Time (s)')
    ax_tl.set_title('TimeSformer v2 — sliding window'); ax_tl.grid(True,alpha=0.3)
    if ev and ev.get('trigger_frame') is not None and isinstance(axes,(list,np.ndarray)):
        fr=cv2.cvtColor(ev['trigger_frame'],cv2.COLOR_BGR2RGB)
        axes[1].imshow(fr)
        if ev['collision_bbox']:
            x1,y1,x2,y2=ev['collision_bbox']
            import matplotlib.patches as mpatches
            axes[1].add_patch(mpatches.Rectangle((x1,y1),x2-x1,y2-y1,
                linewidth=2,edgecolor='yellow',facecolor='none',linestyle='--'))
        axes[1].set_title(f"Trigger frame — accident zone (20% padded bbox)")
        axes[1].axis('off')
    plt.tight_layout(); plt.show()

## Section 5 — Path 2: YOLO11 + ByteTrack

In [ ]:
# 2.1 Load YOLO11n
try:    yolo_track_model = YOLO('yolo11n.pt')
except: yolo_track_model = YOLO('yolov8n.pt'); print("Using yolov8n")
yolo_track_model.to(DEVICE)
print(f"YOLO loaded | VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
# 3.1 Height-normalized VehicleTracker (FIX #3 — scale invariance)
def box_aspect(b):
    w=max(1,b[2]-b[0]); h=max(1,b[3]-b[1]); return w/h

def box_height(b):
    """Bbox height in pixels — used as spatial scale reference."""
    return max(1, b[3]-b[1])

def center_dist(b1,b2):
    c1=((b1[0]+b1[2])/2,(b1[1]+b1[3])/2)
    c2=((b2[0]+b2[2])/2,(b2[1]+b2[3])/2)
    return math.hypot(c1[0]-c2[0],c1[1]-c2[1])

def box_iou(b1,b2):
    ix1=max(b1[0],b2[0]); iy1=max(b1[1],b2[1])
    ix2=min(b1[2],b2[2]); iy2=min(b1[3],b2[3])
    if ix2<=ix1 or iy2<=iy1: return 0.0
    inter=(ix2-ix1)*(iy2-iy1)
    a1=(b1[2]-b1[0])*(b1[3]-b1[1]); a2=(b2[2]-b2[0])*(b2[3]-b2[1])
    return inter/(a1+a2-inter+1e-6)


class VehicleTracker:
    """
    Tracks vehicles using YOLO11 + ByteTrack.

    Speed is stored in TWO units:
      speed_pps   : absolute pixels/sec (for visualization)
      speed_bhps  : bbox-heights/sec (scale-invariant — FIX #3)

    Why bbox-height normalization?
      A car 10m away has bbox_height=80px, moves 300 px/s → 3.75 BH/s
      A car 40m away has bbox_height=20px, moves 75 px/s  → 3.75 BH/s
      Same real-world speed → same normalized speed. Physics is preserved.
    """
    def __init__(self, fps: float, history_len: int = 8):
        self.fps         = fps
        self.hist_len    = history_len
        # per track: deque of (frame_num, cx, cy, bbox_height)
        self.pos_history = defaultdict(lambda: deque(maxlen=history_len))
        self.bh_history  = defaultdict(lambda: deque(maxlen=history_len))
        self.aspect_hist = defaultdict(lambda: deque(maxlen=history_len))

    def update(self, frame_bgr: np.ndarray, frame_num: int):
        results = yolo_track_model.track(frame_bgr, tracker='bytetrack.yaml',
                               classes=list(VEHICLE_IDS), persist=True, verbose=False)
        r = results[0]
        vehicles = []; pedestrians = []
        if r.boxes is None or r.boxes.id is None:
            return vehicles, pedestrians
        for box, tid_t, cls_t, conf_t in zip(
                r.boxes.xyxy, r.boxes.id, r.boxes.cls, r.boxes.conf):
            tid    = int(tid_t.item())
            cls_id = int(cls_t.item())
            x1,y1,x2,y2 = [int(v) for v in box.tolist()]
            cx,cy = (x1+x2)/2, (y1+y2)/2
            bh    = box_height([x1,y1,x2,y2])
            self.pos_history[tid].append((frame_num, cx, cy))
            self.bh_history[tid].append(bh)
            self.aspect_hist[tid].append(box_aspect([x1,y1,x2,y2]))
            # Compute velocity
            h = list(self.pos_history[tid])
            vx_pps=vy_pps=0.0
            if len(h)>=2:
                f0,x0,y0=h[0]; f1,x1h,y1h=h[-1]
                dt=(f1-f0)/self.fps
                if dt>0: vx_pps=(x1h-x0)/dt; vy_pps=(y1h-y0)/dt
            speed_pps  = math.hypot(vx_pps,vy_pps)
            speed_bhps = speed_pps / bh   # bbox-heights per second

            entry = {'id':tid,'class':VEHICLE_CLS.get(cls_id,'vehicle'),
                      'bbox':[x1,y1,x2,y2],'cx':cx,'cy':cy,
                      'speed_pps':round(speed_pps,2),
                      'speed_bhps':round(speed_bhps,3),  # scale-invariant
                      'vx_pps':round(vx_pps,2),'vy_pps':round(vy_pps,2),
                      'conf':round(float(conf_t),3),
                      'bbox_height':bh}
            if cls_id == 0: pedestrians.append(entry)
            else: vehicles.append(entry)
        return vehicles, pedestrians

    def decel_bh_s2(self, tid, window_frames=4):
        """
        Deceleration in bbox-heights/s² (scale-invariant).
        Normal hard brake : ~1-2 BH/s²
        Crash impact      : >4-5 BH/s²
        """
        # Reconstruct BH-normalized speed history
        ph = list(self.pos_history[tid])
        bh_h = list(self.bh_history[tid])
        if len(ph) < window_frames+1 or len(bh_h) < window_frames+1:
            return 0.0
        # speed now (in BH/s)
        f1,x1h,y1h=ph[-1]; f0,x0h,y0h=ph[-window_frames]
        dt=(f1-f0)/self.fps
        if dt<=0: return 0.0
        speed_now  = math.hypot(x1h-x0h,y1h-y0h)/dt / max(1,bh_h[-1])
        # speed before (using earlier window)
        if len(ph)<window_frames*2+1: return 0.0
        f3,x3,y3=ph[-window_frames-1]; f2,x2h,y2h=ph[-window_frames*2]
        dt2=(f3-f2)/self.fps
        if dt2<=0: return 0.0
        speed_prev = math.hypot(x3-x2h,y3-y2h)/dt2 / max(1,bh_h[-window_frames-1])
        # decel = (prev_speed - current_speed) / dt
        return max(0.0, (speed_prev - speed_now) / ((window_frames/self.fps)))

    def aspect_change(self, tid):
        h=list(self.aspect_hist[tid])
        if len(h)<4: return 0.0
        return abs(h[-1]-h[0])/max(h[0],1e-3)

    def ped_speed_spike(self, pid, window_frames=3):
        ph=list(self.pos_history[pid])
        bh_h=list(self.bh_history[pid])
        if len(ph)<window_frames+1: return 0.0
        f1,x1h,y1h=ph[-1]; f0,x0h,y0h=ph[-window_frames]
        dt=(f1-f0)/self.fps
        if dt<=0: return 0.0
        speed_now=math.hypot(x1h-x0h,y1h-y0h)/dt/max(1,bh_h[-1])
        if len(ph)<window_frames*2+1: return speed_now
        f3,x3,y3=ph[-window_frames-1]; f2,x2h,y2h=ph[-window_frames*2]
        dt2=(f3-f2)/self.fps
        if dt2<=0: return speed_now
        speed_prev=math.hypot(x3-x2h,y3-y2h)/dt2/max(1,bh_h[-window_frames-1])
        return max(0.0, speed_now-speed_prev)

print("VehicleTracker v3 ready — height-normalized velocities (BH/s, BH/s²)")
print()
print("Why BH/s²:")
print("  Car 10m away: bbox_h=80px, speed=300px/s → 3.75 BH/s")
print("  Car 40m away: bbox_h=20px, speed=75px/s  → 3.75 BH/s")
print("  Same real physics → same number regardless of distance")

In [ ]:
# 3.2 Physics-calibrated collision detector (BH/s² thresholds, FIX #3+#5)
class FullCollisionDetector:
    """
    All thresholds in BH/s² (bbox-height per second squared).

    Calibration (from road safety physics, normalized to vehicle size):
      Normal hard brake  : ~1.5 BH/s²  (0.8g at typical scales)
      Crash impact       : >4.0 BH/s²  (2g+)
      Severe crash       : >8.0 BH/s²
      Threshold = 3.5 BH/s² — safely between braking and impact

    FIX for low-speed false positives:
      A car at 10 km/h stopping: speed_bhps ≈ 0.4 BH/s to begin with.
      decel = 0.4/0.3s ≈ 1.3 BH/s² → well below 3.5 → NOT flagged.
    """
    # Mode A: vehicle-vehicle
    VV_IOU        = 0.05
    VV_DIST_BH    = 2.0    # collision if within 2 bbox-heights (proximity)
    VV_DECEL      = 3.5    # BH/s²
    VV_MIN_SPEED  = 0.5    # BH/s — must have been moving

    # Mode B: single-vehicle (motorcycle fall, flip)
    SV_DECEL      = 4.0    # BH/s² — stricter for solo events
    SV_MIN_SPEED  = 0.5
    SV_ASPECT_CHG = 0.40
    SV_ALONE_BH   = 4.0    # alone if no vehicle within 4 bbox-heights

    # Mode C: pedestrian strike
    PS_IOU        = 0.03
    PS_VEH_DECEL  = 2.5    # BH/s² — vehicle slows after contact
    PS_VEH_MIN_SPEED = 0.4 # BH/s
    PS_PED_SPIKE  = 2.0    # BH/s spike in pedestrian speed

    # Mode D: static impact (lamppost, barrier)
    SI_DECEL      = 5.0    # BH/s² — very high
    SI_MIN_SPEED  = 1.0    # BH/s — was moving meaningfully
    SI_ALONE_BH   = 5.0    # px / bbox_height

    COOLDOWN = 12

    def __init__(self, fps: float, tracker: VehicleTracker):
        self.fps     = fps
        self.tracker = tracker  # shares history
        self.alerts  = {}       # tid -> last alert frame

    def _ok(self, tid, fn):
        return fn - self.alerts.get(tid, -9999) >= self.COOLDOWN

    def _make_ev(self, fn, ctype, inv, all_v, extra=None):
        cbbox=[min(t['bbox'][0] for t in inv),min(t['bbox'][1] for t in inv),
               max(t['bbox'][2] for t in inv),max(t['bbox'][3] for t in inv)]
        # 20% padding for VLM crop (FIX #4)
        pad=0.20
        W=cbbox[2]-cbbox[0]; H=cbbox[3]-cbbox[1]
        # (will be clamped to frame by VLM notebook)
        cbbox_padded=[int(cbbox[0]-W*pad),int(cbbox[1]-H*pad),
                       int(cbbox[2]+W*pad),int(cbbox[3]+H*pad)]
        ev={'frame_num':fn,'time_sec':round(fn/self.fps,2),
             'collision_type':ctype,
             'vehicle_a':inv[0],'vehicle_b':inv[1] if len(inv)>1 else None,
             'collision_bbox':cbbox,'collision_bbox_padded':cbbox_padded,
             'all_tracks':all_v,
             'iou':box_iou(inv[0]['bbox'],inv[1]['bbox']) if len(inv)>1 else 0.0,
             'decel_a_bhs2':round(self.tracker.decel_bh_s2(inv[0]['id']),3),
             'decel_b_bhs2':round(self.tracker.decel_bh_s2(inv[1]['id']),3) if len(inv)>1 else 0.0}
        if extra: ev.update(extra)
        for t in inv: self.alerts[t['id']]=fn
        return ev

    def check(self, vehicles, pedestrians, frame_num):
        all_t=vehicles+pedestrians

        # ── A: vehicle vs vehicle ─────────────────────────
        for i in range(len(vehicles)):
            for j in range(i+1,len(vehicles)):
                a,b=vehicles[i],vehicles[j]
                if not self._ok(a['id'],frame_num): continue
                iou=box_iou(a['bbox'],b['bbox'])
                dist_bh=center_dist(a['bbox'],b['bbox'])/max(a['bbox_height'],b['bbox_height'],1)
                if iou<self.VV_IOU and dist_bh>self.VV_DIST_BH: continue
                # Must have been moving (avoid parked cars)
                ah=list(self.tracker.pos_history[a['id']])
                bh=list(self.tracker.pos_history[b['id']])
                max_sa=a['speed_bhps'] if len(ah)<3 else max(
                    math.hypot(ah[k+1][1]-ah[k][1],ah[k+1][2]-ah[k][2])/(
                        max(1,(ah[k+1][0]-ah[k][0])/self.fps)*a['bbox_height'])
                    for k in range(len(ah)-1)) if ah else 0
                if max(a['speed_bhps'],b['speed_bhps'],max_sa) < self.VV_MIN_SPEED: continue
                da=self.tracker.decel_bh_s2(a['id']); db=self.tracker.decel_bh_s2(b['id'])
                if max(da,db)<self.VV_DECEL: continue
                return self._make_ev(frame_num,'vehicle_vehicle',[a,b],vehicles,
                                      extra={'decel_bhs2':round(max(da,db),3)})

        # ── C: pedestrian strike ─────────────────────────
        for v in vehicles:
            if not self._ok(v['id'],frame_num): continue
            if v['speed_bhps']<self.PS_VEH_MIN_SPEED: continue
            for p in pedestrians:
                if box_iou(v['bbox'],p['bbox'])<self.PS_IOU: continue
                dv=self.tracker.decel_bh_s2(v['id'])
                sig2a=dv>=self.PS_VEH_DECEL
                ped_spk=self.tracker.ped_speed_spike(p['id'])
                sig2b=ped_spk>=self.PS_PED_SPIKE
                if not(sig2a or sig2b): continue
                return self._make_ev(frame_num,'pedestrian_strike',[v,{**p,'speed_pps':0}],
                                      vehicles,
                                      extra={'veh_decel_bhs2':round(dv,3),
                                             'ped_speed_spike_bhs':round(ped_spk,3),
                                             'signal':'2A' if sig2a else '2B'})

        # ── B: single vehicle anomaly ─────────────────────
        for v in vehicles:
            if not self._ok(v['id'],frame_num): continue
            if v['speed_bhps']<self.SV_MIN_SPEED: continue
            dv=self.tracker.decel_bh_s2(v['id'])
            if dv<self.SV_DECEL: continue
            asp=self.tracker.aspect_change(v['id'])
            is_moto=v['class'] in ('motorcycle','bicycle')
            others=[u for u in vehicles if u['id']!=v['id']]
            alone=all(center_dist(v['bbox'],u['bbox'])/v['bbox_height']>self.SV_ALONE_BH
                       for u in others) if others else True
            if asp>=self.SV_ASPECT_CHG: ctype='vehicle_flip'
            elif is_moto and alone:       ctype='motorcycle_fall'
            elif alone:                   ctype='single_vehicle_anomaly'
            else: continue
            return self._make_ev(frame_num,ctype,[v],vehicles,
                                  extra={'decel_bhs2':round(dv,3),
                                         'aspect_change':round(asp,3),'alone':alone})

        # ── D: static impact ──────────────────────────────
        for v in vehicles:
            if not self._ok(v['id'],frame_num): continue
            if v['speed_bhps']<self.SI_MIN_SPEED: continue
            dv=self.tracker.decel_bh_s2(v['id'])
            if dv<self.SI_DECEL: continue
            others=[u for u in vehicles if u['id']!=v['id']]
            if any(center_dist(v['bbox'],u['bbox'])/v['bbox_height']<self.SI_ALONE_BH
                    for u in others): continue
            return self._make_ev(frame_num,'static_impact',[v],vehicles,
                                  extra={'decel_bhs2':round(dv,3)})
        return None

print("FullCollisionDetector v3 ready (BH/s² thresholds)")
print()
print("Thresholds:")
d=FullCollisionDetector
print(f"  Vehicle-vehicle : >{d.VV_DECEL} BH/s²  (proximity: <{d.VV_DIST_BH} BH apart)")
print(f"  Single vehicle  : >{d.SV_DECEL} BH/s²  + aspect change OR alone")
print(f"  Ped decel       : >{d.PS_VEH_DECEL} BH/s²  OR ped spike >{d.PS_PED_SPIKE} BH/s")
print(f"  Static impact   : >{d.SI_DECEL} BH/s²  alone")
print()
print("Car stops gently from 10 km/h (was 89% percentage flag):")
print("  speed_bhps ≈ 0.4 BH/s → decel ≈ 1.3 BH/s² → NOT flagged ✅")

In [ ]:
# 3.3 Diagnostic — confirm vehicle detection + measure actual values
# Checks: (1) YOLO sees vehicles, (2) speeds are non-zero, (3) IoU at crash moment

import numpy as np

vids = sorted(VIDEOS_DIR.glob('*.mp4'))
print(f"Videos available: {len(vids)}")
if not vids:
    print("ERROR: No videos found. Check VIDEOS_DIR path.")
else:
    # Quick detection check on first video (first 10 seconds)
    vid = vids[0]
    cap = cv2.VideoCapture(str(vid))
    fps_v = cap.get(cv2.CAP_PROP_FPS) or 30.0
    limit = min(int(cap.get(cv2.CAP_PROP_FRAME_COUNT)), int(10*fps_v))
    print(f"\nChecking {vid.name} (first 10s @ {fps_v:.0f}fps)")

    trk = VehicleTracker(fps=fps_v)
    counts=[]; max_iou=0.0; max_spd=0.0; max_decel=0.0
    iou_events=[]

    for fn in range(limit):
        ret,frame=cap.read()
        if not ret: break
        vehicles,peds=trk.update(frame,fn)
        counts.append(len(vehicles))
        for v in vehicles:
            max_spd=max(max_spd, v['speed_bhps'])
        # Check all pairs for IoU
        for i in range(len(vehicles)):
            for j in range(i+1,len(vehicles)):
                iou=box_iou(vehicles[i]['bbox'],vehicles[j]['bbox'])
                dist_bh=center_dist(vehicles[i]['bbox'],vehicles[j]['bbox'])/max(vehicles[i]['bbox_height'],1)
                da=trk.decel_bh_s2(vehicles[i]['id'])
                db=trk.decel_bh_s2(vehicles[j]['id'])
                if iou>max_iou: max_iou=iou
                if max(da,db)>max_decel: max_decel=max(da,db)
                if iou>0.01 or dist_bh<3.0:
                    iou_events.append({'fn':fn,'iou':round(iou,4),'dist_bh':round(dist_bh,2),
                                        'decel':round(max(da,db),3)})
    cap.release()

    print(f"\n  Vehicle detection:")
    print(f"    Frames checked        : {len(counts)}")
    print(f"    Avg vehicles/frame    : {sum(counts)/max(len(counts),1):.1f}")
    print(f"    Max vehicles in frame : {max(counts) if counts else 0}")
    print(f"    Frames with 0 vehicles: {counts.count(0)}")
    print(f"\n  Physics values seen:")
    print(f"    Max speed_bhps        : {max_spd:.3f} BH/s")
    print(f"    Max decel seen        : {max_decel:.3f} BH/s²")
    print(f"    Max IoU between pairs : {max_iou:.4f}")
    print(f"    Pairs within 3 BH     : {len(iou_events)}")
    if iou_events:
        top=sorted(iou_events,key=lambda x:-x['iou'])[:5]
        print(f"\n  Top 5 closest pairs:")
        print(f"    {'frame':>6} {'iou':>7} {'dist_bh':>8} {'decel':>7}")
        for e in top:
            print(f"    {e['fn']:>6} {e['iou']:>7.4f} {e['dist_bh']:>8.2f} {e['decel']:>7.3f}")
    print(f"\n  Current thresholds:")
    print(f"    VV_IOU    : {FullCollisionDetector.VV_IOU}  (IoU contact — now 0.02)")
    print(f"    VV_DIST_BH: {FullCollisionDetector.VV_DIST_BH}  (proximity in BH)")
    print(f"    VV_DECEL  : {FullCollisionDetector.VV_DECEL}  BH/s²")
    if max(counts,default=0)==0:
        print("\n  ⚠ YOLO detected NO vehicles — check model loaded correctly (cell 2.1)")
    elif max_iou < 0.02:
        print(f"\n  No direct vehicle contact in first 10s (max IoU={max_iou:.4f})")
        print(f"  This is normal — crash happens at a specific moment in the video.")
        print(f"  New A1 trigger (iou>=0.02) will catch it when it occurs.")
    else:
        print(f"\n  ✅ IoU events detected — new A1 trigger will flag these")


# 4.1 Full video pipeline
def process_video(video_path, max_sec=120, visualize=True):
    video_path = Path(video_path)
    cap   = cv2.VideoCapture(str(video_path))
    fps   = cap.get(cv2.CAP_PROP_FPS) or 30.0
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    limit = min(total, int(max_sec * fps))
    h_f   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    w_f   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))

    tracker  = VehicleTracker(fps=fps)
    detector = FullCollisionDetector(fps=fps, tracker=tracker)
    collisions = []; timeline = []; snap = None

    bar = tqdm(
        total=limit,
        desc=f"{video_path.name[:28]}",
        dynamic_ncols=True,
        leave=True,
        bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}] {postfix}'
    )

    for fn in range(limit):
        ret, frame = cap.read()
        if not ret: break

        vehicles, peds = tracker.update(frame, fn)
        all_tracks     = vehicles + peds
        event          = detector.check(vehicles, peds, fn)
        max_bh         = max((t['speed_bhps'] for t in vehicles), default=0)

        timeline.append({'frame': fn, 'time': fn/fps,
                          'n_v': len(vehicles), 'max_bhps': max_bh})

        if event:
            collisions.append(event)
            if snap is None: snap = frame.copy()
            vb     = event['vehicle_b']
            vb_str = f"+ {vb['class']}(ID{vb['id']})" if vb else "(single vehicle)"
            tqdm.write(
                f"  🚨 [{event['collision_type']:<22}] "
                f"t={event['time_sec']:.1f}s | "
                f"{event['vehicle_a']['class']}(ID{event['vehicle_a']['id']}) {vb_str} | "
                f"decel={event['decel_a_bhs2']:.2f} BH/s²"
            )

        # Update bar every 30 frames to avoid slowdown
        if fn % 30 == 0:
            bar.set_postfix(
                vehicles=len(vehicles),
                events=len(collisions),
                spd=f"{max_bh:.1f}BH/s",
                refresh=False
            )
        bar.update(1)

    bar.set_postfix(vehicles='—', events=len(collisions), spd='—')
    bar.close()
    cap.release()

    print(f"  Total events: {len(collisions)}")

    if visualize and collisions and snap is not None:
        ev = collisions[0]
        fig, axes = plt.subplots(1, 2, figsize=(16, 6))
        axes[0].imshow(cv2.cvtColor(snap, cv2.COLOR_BGR2RGB))
        for t in ev['all_tracks']:
            x1, y1, x2, y2 = t['bbox']
            inv = t['id'] in (
                [ev['vehicle_a']['id']] +
                ([ev['vehicle_b']['id']] if ev['vehicle_b'] else []))
            c  = '#ff2222' if inv else '#3498db'
            lw = 3 if inv else 1
            axes[0].add_patch(patches.Rectangle(
                (x1, y1), x2-x1, y2-y1,
                linewidth=lw, edgecolor=c, facecolor='none'))
            axes[0].text(x1, y1-4, f"ID{t['id']} {t['speed_bhps']:.1f}BH/s",
                color='white', fontsize=7,
                bbox=dict(facecolor=c, alpha=0.7, pad=1, edgecolor='none'))
        pb   = ev['collision_bbox_padded']
        pb_c = [max(0,pb[0]), max(0,pb[1]), min(w_f,pb[2]), min(h_f,pb[3])]
        axes[0].add_patch(patches.Rectangle(
            (pb_c[0], pb_c[1]), pb_c[2]-pb_c[0], pb_c[3]-pb_c[1],
            linewidth=2, edgecolor='#ffff00', facecolor='none',
            linestyle='--', label='VLM crop (20% padded)'))
        axes[0].set_title(
            f"[{ev['collision_type']}] at {ev['time_sec']:.1f}s | yellow = VLM crop")
        axes[0].axis('off')

        axes[1].plot([t['time'] for t in timeline],
                      [t['max_bhps'] for t in timeline],
                      'b-', lw=1.5, label='max speed (BH/s)')
        for ev2 in collisions:
            axes[1].axvline(ev2['time_sec'], color='red', ls='--',
                             lw=1.5, alpha=0.7, label=ev2['collision_type'])
        axes[1].set_xlabel('Time (s)'); axes[1].set_ylabel('Max speed (BH/s)')
        axes[1].set_title('Scale-invariant speed timeline')
        axes[1].grid(True, alpha=0.3)
        handles, labels = axes[1].get_legend_handles_labels()
        seen = set(); uhan = []; ulab = []
        for h_, l_ in zip(handles, labels):
            if l_ not in seen: seen.add(l_); uhan.append(h_); ulab.append(l_)
        axes[1].legend(uhan, ulab, fontsize=8)
        plt.tight_layout(); plt.show()

    return collisions, timeline, snap


vids = sorted(VIDEOS_DIR.glob('*.mp4'))
if vids:
    events, tl, snap = process_video(vids[0])
else:
    print("No videos found")

In [ ]:
# 4.2 Batch evaluation
def run_batch(vids, n=20):
    results=[]
    for vf in tqdm(vids[:n], desc="Batch", position=0, leave=True, dynamic_ncols=True,
                   bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}]'):
        try:
            evs,tl,_=process_video(vf,visualize=False)
            n_ev=len(evs)
            types=[e['collision_type'] for e in evs]
            # Single summary line per video
            tqdm.write(f"  {vf.name[:35]:<35} events={n_ev}  {types if types else ''}")
            results.append({'video':vf.name,'n_events':n_ev,'types':types})
        except Exception as ex:
            tqdm.write(f"  ERROR {vf.name}: {ex}")
    return results

vids=sorted(VIDEOS_DIR.glob('*.mp4'))
print(f"Running batch on first 20 of {len(vids)} videos...")
batch=run_batch(vids,20)
from collections import Counter
all_types=Counter(t for r in batch for t in r['types'])
print(f"  Videos with events : {sum(1 for r in batch if r['n_events']>0)}/{len(batch)}")
print(f"  Events by type     : {dict(all_types)}")
out=RESULTS_DIR/'bytetrack_v3_results.json'
with open(out,'w') as f: json.dump(batch,f,indent=2)
print(f"Saved: {out}")

# 5.1 VLM integration export
def extract_accident_context(event, frame_bgr, frame_w, frame_h):
    """
    Package event for VLM_Triage_Qwen25_v2.ipynb.
    Adds physics data from the BH/s² tracker.
    collision_bbox_padded already has 20% padding for VLM crop.
    """
    inv=[event['vehicle_a']] + ([event['vehicle_b']] if event['vehicle_b'] else [])
    pb=event.get('collision_bbox_padded',event['collision_bbox'])
    pb_clamped=[max(0,pb[0]),max(0,pb[1]),min(frame_w,pb[2]),min(frame_h,pb[3])]
    return {
        'involved_vehicles':  inv,
        'vehicle_types':      list({v['class'] for v in inv}),
        'vehicle_count':      len(inv),
        'pedestrian_count':   sum(1 for t in event['all_tracks'] if t['class']=='person'),
        'max_vehicle_overlap': event['iou'],
        'collision_point':    ((inv[0]['cx']+(inv[1]['cx'] if len(inv)>1 else inv[0]['cx']))/2,
                               (inv[0]['cy']+(inv[1]['cy'] if len(inv)>1 else inv[0]['cy']))/2),
        'accident_bbox':      pb_clamped,   # 20%-padded, clamped to frame
        'all_vehicles_count': len(event['all_tracks']),
        'collision_type':     event['collision_type'],
        'detections':         event['all_tracks'],
        '_physics': {
            'decel_a_bhs2':    event['decel_a_bhs2'],
            'decel_b_bhs2':    event['decel_b_bhs2'],
            'speed_a_bhps':    event['vehicle_a']['speed_bhps'],
            'speed_b_bhps':    event['vehicle_b']['speed_bhps'] if event['vehicle_b'] else 0.0,
            'direct_contact':  event['iou'] > 0.05,
        }
    }

if 'events' in dir() and events and snap is not None:
    h_f,w_f=snap.shape[:2]
    ctx=extract_accident_context(events[0],snap,w_f,h_f)
    print("VLM context ready:")
    print(f"  Collision type   : {ctx['collision_type']}")
    print(f"  Vehicles         : {ctx['vehicle_types']}")
    print(f"  Padded bbox      : {ctx['accident_bbox']}")
    print(f"  Physics          : {ctx['_physics']}")
    print()
    print("Pass this as yolo_data to generate_triage_report() in VLM notebook.")


## Section 6 — VLM Triage: Qwen2.5-VL-7B-AWQ
⚠️ **Run the VRAM cleanup cell before loading Qwen**

In [ ]:
import gc
# Free TimeSformer and any other models from memory
for var in ['timesformer_model', 'effnet_model', 'yolo_cls_model', 'model']:
    if var in dir():
        del globals()[var]
gc.collect()
torch.cuda.empty_cache()
print(f"VRAM freed. Available: {torch.cuda.memory_reserved()/1e9:.1f} GB reserved")

In [ ]:
# ============================================================
# 2.1  Load YOLOv8n vehicle detector
# ============================================================
# Uses the same model already downloaded in the EfficientNet
# pipeline — no extra download needed if cached.
# ============================================================

VEHICLE_CLASSES = {
    0: 'person',
    1: 'bicycle',
    2: 'car',
    3: 'motorcycle',
    5: 'bus',
    7: 'truck'
}
VEHICLE_IDS = set(VEHICLE_CLASSES.keys())

yolo = YOLO('yolov8n.pt')
yolo.to(DEVICE)

print("✅ YOLOv8n loaded")
print(f"   Detecting: {list(VEHICLE_CLASSES.values())}")
print(f"   VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
# ============================================================
# 2.2  Load Qwen2.5-VL-7B-Instruct (BitsAndBytes NF4 Quantization)
# ============================================================
# Uses NF4 dynamic quantization to bypass the AWQ ExLlamaV2 crash
# on the 3420 vision dimension. Fits perfectly in Colab T4.
# ============================================================


# SANITY CHECK: Free VRAM if re-running
if 'vlm_model' in globals():
    print("🧹 Cleaning up existing model instance to free VRAM...")
    del vlm_model
    if 'vlm_processor' in globals():
        del vlm_processor
    gc.collect()
    torch.cuda.empty_cache()

# Use the BASE model, not the AWQ version
MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"

print(f"Loading {MODEL_ID} with BitsAndBytes NF4...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

t0 = time.time()

vlm_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    device_map="auto",
    quantization_config=bnb_config
)

vlm_processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=256 * 28 * 28,
    max_pixels=512 * 28 * 28,
)

elapsed = time.time() - t0
print(f"\n✅ Qwen2.5-VL-7B (NF4) loaded in {elapsed:.0f}s")
print(f"   VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB")
print(f"   VRAM free: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")

In [ ]:
# ============================================================
# 3.1  Vehicle detection — with accident cluster isolation
# ============================================================
import cv2
import time
import torch
from PIL import Image

VEHICLE_CLASSES = {
    0: 'person', 1: 'bicycle', 2: 'car',
    3: 'motorcycle', 5: 'bus', 7: 'truck'
}
VEHICLE_IDS = set(VEHICLE_CLASSES.keys())
CLUSTER_RADIUS_FRAC = 0.30  # fraction of frame width

def _box_center(bbox):
    return ((bbox[0]+bbox[2])/2, (bbox[1]+bbox[3])/2)

def _box_iou(b1, b2):
    ix1=max(b1[0],b2[0]); iy1=max(b1[1],b2[1])
    ix2=min(b1[2],b2[2]); iy2=min(b1[3],b2[3])
    if ix2<=ix1 or iy2<=iy1: return 0.0
    inter=(ix2-ix1)*(iy2-iy1)
    a1=(b1[2]-b1[0])*(b1[3]-b1[1])
    a2=(b2[2]-b2[0])*(b2[3]-b2[1])
    return inter/(a1+a2-inter+1e-6)

def _center_dist(b1, b2):
    c1,c2=_box_center(b1),_box_center(b2)
    return ((c1[0]-c2[0])**2+(c1[1]-c2[1])**2)**0.5

def find_accident_cluster(detections, frame_w):
    vehicles    = [d for d in detections if d['class'] != 'person']
    pedestrians = [d for d in detections if d['class'] == 'person']

    if len(vehicles) == 0:
        return {'involved_vehicles':[],'all_vehicles':[],'pedestrians_near':[],
                'pedestrians_all':pedestrians,'collision_point':None,
                'max_iou':0.0,'accident_bbox':None}

    if len(vehicles) == 1:
        v=vehicles[0]; cx,cy=_box_center(v['bbox'])
        return {'involved_vehicles':vehicles,'all_vehicles':vehicles,
                'pedestrians_near':[],'pedestrians_all':pedestrians,
                'collision_point':(cx,cy),'max_iou':0.0,'accident_bbox':v['bbox']}

    best_iou=0.0; best_pair=(0,1); best_dist=float('inf'); closest_pair=(0,1)
    for i in range(len(vehicles)):
        for j in range(i+1, len(vehicles)):
            iou  = _box_iou(vehicles[i]['bbox'], vehicles[j]['bbox'])
            dist = _center_dist(vehicles[i]['bbox'], vehicles[j]['bbox'])
            if iou  > best_iou:  best_iou=iou;   best_pair=(i,j)
            if dist < best_dist: best_dist=dist;  closest_pair=(i,j)

    anchor  = best_pair if best_iou > 0.05 else closest_pair
    v_a,v_b = vehicles[anchor[0]], vehicles[anchor[1]]
    ca,cb   = _box_center(v_a['bbox']), _box_center(v_b['bbox'])
    cx,cy   = (ca[0]+cb[0])/2, (ca[1]+cb[1])/2
    radius  = frame_w * CLUSTER_RADIUS_FRAC

    involved = [v for v in vehicles
                if _center_dist(v['bbox'], [cx,cy,cx,cy]) <= radius
                or v is v_a or v is v_b]

    if involved:
        ax1=min(v['bbox'][0] for v in involved); ay1=min(v['bbox'][1] for v in involved)
        ax2=max(v['bbox'][2] for v in involved); ay2=max(v['bbox'][3] for v in involved)
        accident_bbox=[ax1,ay1,ax2,ay2]
    else:
        involved=[v_a,v_b]; accident_bbox=None

    peds_near=[p for p in pedestrians
               if _center_dist(p['bbox'],[cx,cy,cx,cy]) <= radius*1.5]

    return {'involved_vehicles':involved,'all_vehicles':vehicles,
            'pedestrians_near':peds_near,'pedestrians_all':pedestrians,
            'collision_point':(round(cx,1),round(cy,1)),
            'max_iou':round(best_iou,3),'accident_bbox':accident_bbox}

def detect_vehicles(frame_bgr, conf=0.25):
    h,w = frame_bgr.shape[:2]
    results = yolo.predict(frame_bgr, verbose=False, conf=conf,
                           classes=list(VEHICLE_IDS))
    r = results[0]
    detections=[]
    if r.boxes is not None and len(r.boxes)>0:
        for box in r.boxes:
            cls_id=int(box.cls.item())
            if cls_id not in VEHICLE_IDS: continue
            x1,y1,x2,y2=[int(v) for v in box.xyxy[0].tolist()]
            detections.append({'class':VEHICLE_CLASSES[cls_id],
                               'confidence':round(float(box.conf.item()),3),
                               'bbox':[x1,y1,x2,y2]})
    cluster=find_accident_cluster(detections, frame_w=w)
    type_counts={}
    for v in cluster['involved_vehicles']:
        c=v['class']; type_counts[c]=type_counts.get(c,0)+1
    return {'involved_vehicles':cluster['involved_vehicles'],
            'vehicle_types':list(type_counts.keys()),
            'vehicle_count':len(cluster['involved_vehicles']),
            'pedestrian_count':len(cluster.get('pedestrians_near',[])),
            'max_vehicle_overlap':cluster['max_iou'],
            'collision_point':cluster['collision_point'],
            'accident_bbox':cluster['accident_bbox'],
            'all_vehicles_count':len(cluster['all_vehicles']),
            'detections':detections}

print("✅ Vehicle detection with accident cluster isolation ready")



# ============================================================
# 3.2  VLM triage prompt — grounded, enum hazards, no hallucination
# ============================================================

ALLOWED_HAZARDS = [
    "vehicle_blocking_lane", "debris_on_road", "fluid_spill",
    "smoke_visible", "fire_visible", "pedestrian_near_collision",
    "overturned_vehicle", "airbag_deployed", "broken_glass_visible",
    "traffic_backup",
]

SYSTEM_PROMPT = """You are an AI emergency response assistant analyzing
a traffic camera frame after an accident has been detected.

CRITICAL RULES:
1. Output ONLY valid JSON — no markdown, no code fences, no explanation.
2. Every field must be present. Use null if unknown.
3. HAZARDS: Only use values from this exact list —
   vehicle_blocking_lane | debris_on_road | fluid_spill |
   smoke_visible | fire_visible | pedestrian_near_collision |
   overturned_vehicle | airbag_deployed | broken_glass_visible |
   traffic_backup
   Use empty list [] if none are visually confirmed.
4. DO NOT infer hazards. Only report what you can see.
   Cannot clearly see flames? Do NOT add fire_visible.
   Cannot clearly see smoke? Do NOT add smoke_visible.
5. vehicles_involved: list ONLY vehicles at the collision zone.
   Ignore vehicles clearly far from the accident point.
6. recommended_units: always include police. Add ambulance if severity>=3.
   Add fire ONLY if fire_visible is confirmed in hazards.

JSON schema:
{
  "alert_level": "LOW | MEDIUM | HIGH | CRITICAL",
  "scene_description": "one factual sentence",
  "vehicles_involved": ["car","motorcycle",etc — involved only],
  "vehicle_count": <integer>,
  "pedestrians_in_scene": <true|false>,
  "collision_zone": "location description",
  "road_type": "intersection|highway|urban road|rural road|unknown",
  "hazards": ["from allowed list only"],
  "visibility_conditions": "daytime|nighttime|low light|rain|fog|unknown",
  "severity_score": <1-5>,
  "recommended_units": ["police"] or ["police","ambulance"] etc,
  "confidence": <0.0-1.0>
}

Severity: 1=minor 2=moderate 3=serious 4=major 5=critical"""

def build_user_message(frame_pil, yolo_data):
    inv   = yolo_data.get('involved_vehicles', [])
    total = yolo_data.get('all_vehicles_count', 0)
    cp    = yolo_data.get('collision_point')
    abbox = yolo_data.get('accident_bbox')
    types = ', '.join(v['class'] for v in inv) if inv else 'not detected'
    zone  = (f"pixel [{abbox[0]},{abbox[1]}] to [{abbox[2]},{abbox[3]}]"
             if abbox else f"approx ({cp[0]:.0f},{cp[1]:.0f})" if cp else "unknown")
    context = (
        f"YOLOv8n results:\n"
        f"  Vehicles in accident zone: {len(inv)} ({types})\n"
        f"  Total vehicles in frame  : {total} (ignore distant ones)\n"
        f"  Accident zone location   : {zone}\n"
        f"  Vehicle overlap IoU      : {yolo_data.get('max_vehicle_overlap',0):.3f}"
        f" ({'direct contact' if yolo_data.get('max_vehicle_overlap',0)>0.05 else 'near miss/close'})\n"
        f"  Pedestrians near zone    : {yolo_data.get('pedestrian_count',0)}\n"
        f"\nFocus on the accident zone. Output triage JSON only."
    )
    return [
        # FIXED: Properly inject the SYSTEM_PROMPT into the messages array
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
        {"role": "user", "content": [
            {"type": "image", "image": frame_pil},
            {"type": "text", "text": context}
        ]}
    ]

print("✅ Grounded triage prompt ready")


print(f"   Closed hazard enum: {len(ALLOWED_HAZARDS)} allowed values")


In [ ]:
# ============================================================
# 3.3  Inference + post-processing validation
# ============================================================
import re as _re, json as _json

def _check_fire_smoke(frame_bgr, accident_bbox):
    """Check if fire/smoke colors actually present in accident zone."""
    if accident_bbox:
        x1,y1,x2,y2 = accident_bbox
        pad=30; h,w=frame_bgr.shape[:2]
        zone=frame_bgr[max(0,y1-pad):min(h,y2+pad),
                       max(0,x1-pad):min(w,x2+pad)]
    else:
        zone=frame_bgr
    if zone.size==0:
        return {'fire_likely':False,'smoke_likely':False}
    hsv = cv2.cvtColor(zone,cv2.COLOR_BGR2HSV).astype(float)
    H,S,V = hsv[:,:,0],hsv[:,:,1],hsv[:,:,2]
    total  = H.size
    fire_r  = float(((( H<20)|(H>160))&(S>120)&(V>80) ).sum()) / total
    smoke_r = float(((S<40)&(V>150)).sum()) / total
    return {'fire_likely':fire_r>0.02,'smoke_likely':smoke_r>0.05,
            'fire_ratio':round(fire_r,4),'smoke_ratio':round(smoke_r,4)}

def validate_report(report, frame_bgr, yolo_data):
    """Remove hallucinated fire/smoke if pixels don't support them."""
    hazards = report.get('hazards', [])
    if not isinstance(hazards, list): hazards=[]
    fire_c  = 'fire_visible'  in hazards
    smoke_c = 'smoke_visible' in hazards
    if fire_c or smoke_c:
        pix = _check_fire_smoke(frame_bgr, yolo_data.get('accident_bbox'))
        report['_pixel_check'] = pix
        removed=[]
        if fire_c  and not pix['fire_likely']:
            hazards=[h for h in hazards if h!='fire_visible'];  removed.append('fire_visible')
        if smoke_c and not pix['smoke_likely']:
            hazards=[h for h in hazards if h!='smoke_visible']; removed.append('smoke_visible')
        if removed:
            report['_removed_hallucinations']=removed
            if 'fire_visible' in removed:
                units=report.get('recommended_units',[])
                if isinstance(units,list):
                    report['recommended_units']=[u for u in units if u!='fire']
    # Clamp to allowed list
    report['hazards']=[h for h in hazards if h in ALLOWED_HAZARDS]
    return report

def _parse_json_robust(text):
    # SAFE FIX: Uses simple string replacement instead of regex backticks!
    clean_text = text.replace("```json", "").replace("```", "").strip()
    try:
        return _json.loads(clean_text)
    except _json.JSONDecodeError:
        m=_re.search(r'\{.*\}',clean_text,_re.DOTALL)
        if m:
            try: return _json.loads(m.group())
            except: pass
    return {'_parse_error':True,'_raw':text}

from qwen_vl_utils import process_vision_info

def generate_triage_report(frame_bgr, max_new_tokens=220, temperature=0.1):
    t_start=time.time()
    t1=time.time(); yolo_data=detect_vehicles(frame_bgr); t_yolo=time.time()-t1
    frame_pil=Image.fromarray(cv2.cvtColor(frame_bgr,cv2.COLOR_BGR2RGB))
    messages=build_user_message(frame_pil, yolo_data)
    text=vlm_processor.apply_chat_template(messages,tokenize=False,add_generation_prompt=True)
    image_inputs,video_inputs=process_vision_info(messages)
    inputs=vlm_processor(text=[text],images=image_inputs,videos=video_inputs,
                         padding=True,return_tensors="pt").to(DEVICE)
    t2=time.time()
    with torch.no_grad():
        output_ids=vlm_model.generate(**inputs,max_new_tokens=max_new_tokens,
                                       temperature=temperature,do_sample=(temperature>0))
    trimmed=[o[len(i):] for i,o in zip(inputs.input_ids,output_ids)]
    raw=vlm_processor.batch_decode(trimmed,skip_special_tokens=True,
                                    clean_up_tokenization_spaces=False)[0].strip()
    t_vlm=time.time()-t2
    report=_parse_json_robust(raw)
    report=validate_report(report,frame_bgr,yolo_data)
    report['_raw_output']=raw
    report['_yolo_data']=yolo_data
    report['_timing']={'yolo_ms':round(t_yolo*1000,1),'vlm_sec':round(t_vlm,2),
                        'total_sec':round(time.time()-t_start,2),
                        'tokens_gen':len(trimmed[0]) if trimmed else 0}
    return report

print("✅ Inference + fire/smoke validation ready (Colab syntax error fixed)")

# ============================================================
# 3.4  Visualization helper
# ============================================================

ALERT_COLORS = {
    'LOW':      '#2ecc71',
    'MEDIUM':   '#f39c12',
    'HIGH':     '#e74c3c',
    'CRITICAL': '#8e44ad',
}

CLASS_COLORS = {
    'car': '#3498db', 'truck': '#e74c3c', 'bus': '#9b59b6',
    'motorcycle': '#f39c12', 'bicycle': '#2ecc71', 'person': '#1abc9c',
}

def visualize_triage(frame_bgr: np.ndarray, report: dict, title: str = ""):
    """Display frame with YOLO boxes and triage report side by side."""
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    h, w = frame_rgb.shape[:2]

    fig = plt.figure(figsize=(18, 8))
    gs  = fig.add_gridspec(1, 2, width_ratios=[1.4, 1])
    ax_img  = fig.add_subplot(gs[0])
    ax_text = fig.add_subplot(gs[1])

    # ── Left: Frame with YOLO boxes ─────────────────────────
    ax_img.imshow(frame_rgb)
    yolo_data = report.get('_yolo_data', {})
    for det in yolo_data.get('detections', []):
        x1, y1, x2, y2 = det['bbox']
        color = CLASS_COLORS.get(det['class'], '#ffffff')
        rect = patches.Rectangle(
            (x1, y1), x2-x1, y2-y1,
            linewidth=2, edgecolor=color, facecolor='none'
        )
        ax_img.add_patch(rect)
        ax_img.text(
            x1, y1-6,
            f"{det['class']} {det['confidence']:.0%}",
            color='white', fontsize=8,
            bbox=dict(facecolor=color, alpha=0.8, pad=2, edgecolor='none')
        )

    alert = report.get('alert_level', 'UNKNOWN')
    alert_color = ALERT_COLORS.get(alert, '#888888')
    ax_img.set_title(
        f"⚠ ALERT: {alert}  |  Severity: {report.get('severity_score', '?')}/5",
        fontsize=13, color=alert_color, fontweight='bold', pad=10
    )
    ax_img.axis('off')

    # ── Right: Triage report ────────────────────────────────
    ax_text.axis('off')
    lines = []

    def add(label, value, indent=0):
        prefix = "  " * indent
        if value is None:
            lines.append(f"{prefix}{label}: —")
        elif isinstance(value, list):
            if value:
                lines.append(f"{prefix}{label}:")
                for v in value:
                    lines.append(f"{prefix}  • {v}")
            else:
                lines.append(f"{prefix}{label}: none")
        else:
            lines.append(f"{prefix}{label}: {value}")

    lines.append("─" * 38)
    lines.append(" TRIAGE REPORT")
    lines.append("─" * 38)
    add("Scene",     report.get('scene_description'))
    lines.append("")
    add("Road type",     report.get('road_type'))
    add("Collision zone",report.get('collision_zone'))
    add("Visibility",    report.get('visibility_conditions'))
    lines.append("")
    add("Vehicles",      report.get('vehicles_involved'))
    add("Pedestrians",   report.get('pedestrians_in_scene'))
    lines.append("")
    add("Hazards",       report.get('hazards'))
    lines.append("")
    add("Units needed",  report.get('recommended_units'))
    lines.append("")

    timing = report.get('_timing', {})
    lines.append("─" * 38)
    lines.append(f" YOLO:  {timing.get('yolo_ms', '?')} ms")
    lines.append(f" VLM:   {timing.get('vlm_sec', '?')} sec")
    lines.append(f" Total: {timing.get('total_sec', '?')} sec")
    lines.append(f" Confidence: {report.get('confidence', '?')}")
    lines.append("─" * 38)

    text = "\n".join(lines)
    ax_text.text(
        0.02, 0.98, text,
        transform=ax_text.transAxes,
        verticalalignment='top',
        fontfamily='monospace',
        fontsize=9.5,
        color='#1a1a2e',
        bbox=dict(boxstyle='round,pad=0.8', facecolor=f'{alert_color}22',
                  edgecolor=alert_color, linewidth=1.5)
    )

    if title:
        fig.suptitle(title, fontsize=12, y=1.01)
    plt.tight_layout()
    plt.show()
    return fig

print("✅ Visualization helper ready")

In [ ]:
# ============================================================
# 4.1  Test on accident frames from dataset
# ============================================================
# Loads sample accident frames already extracted by the
# EfficientNet pipeline and runs the full triage pipeline.
# ============================================================

acc_dir = DATASET_DIR / 'test' / 'accident'
if not acc_dir.exists():
    print(f"❌ Accident frames not found at {acc_dir}")
    print("   Run the EfficientNet notebook first to build the dataset.")
else:
    import random
    sample_paths = random.sample(
        list(acc_dir.glob('*.jpg')),
        min(3, len(list(acc_dir.glob('*.jpg'))))
    )

    print(f"Testing on {len(sample_paths)} accident frames from dataset\n")

    reports = []
    for i, img_path in enumerate(sample_paths):
        print(f"── Frame {i+1}/{len(sample_paths)}: {img_path.name} ──")

        frame_bgr = cv2.imread(str(img_path))
        if frame_bgr is None:
            print("  ❌ Could not read frame, skipping")
            continue

        report = generate_triage_report(frame_bgr)
        reports.append({'path': str(img_path), 'report': report})

        # Print timing
        t = report['_timing']
        print(f"  ✅ Done — YOLO: {t['yolo_ms']}ms | VLM: {t['vlm_sec']}s | Total: {t['total_sec']}s")
        print(f"  Alert: {report.get('alert_level','?')} | Severity: {report.get('severity_score','?')}/5")
        print(f"  Scene: {report.get('scene_description','?')[:80]}")
        print()

        visualize_triage(frame_bgr, report, title=img_path.name)

    # Save results
    out_file = RESULTS_DIR / 'dataset_test_reports.json'
    with open(out_file, 'w') as f:
        # Remove numpy arrays before saving
        clean = []
        for r in reports:
            rc = dict(r)
            rc['report'] = {k: v for k, v in r['report'].items()
                           if not k.startswith('_yolo') and k != '_raw_output'}
            clean.append(rc)
        json.dump(clean, f, indent=2)
    print(f"💾 Reports saved to {out_file}")

# ============================================================
# 4.2  Test on a real video — detect accident, trigger triage
# ============================================================
# Simulates the full cascade:
# 1. Slide through video frames
# 2. Detect motion threshold (proxy for EfficientNet)
# 3. On first candidate frame, run full triage pipeline
# ============================================================

def run_triage_on_video(
    video_path:    str,
    sample_every:  int   = 30,   # check every N frames (~1s at 30fps)
    motion_thresh: float = 0.15, # pixel change fraction to flag candidate
    max_sec:       int   = 120,
) -> dict | None:
    """
    Simple demo: uses frame-difference motion detection as a trigger.
    In the full system, EfficientNet replaces this trigger.
    Runs triage only on the first high-motion frame found.
    """
    video_path = Path(video_path)
    cap   = cv2.VideoCapture(str(video_path))
    fps   = cap.get(cv2.CAP_PROP_FPS) or 30
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    limit = min(total, int(max_sec * fps))

    print(f"🎬 {video_path.name}: {total} frames @ {fps:.0f}fps")

    prev_gray = None
    best_frame = None
    best_score = 0.0

    for i in range(limit):
        ret, frame = cap.read()
        if not ret:
            break
        if i % sample_every != 0:
            continue

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        gray = cv2.GaussianBlur(gray, (21, 21), 0)

        if prev_gray is not None:
            diff  = cv2.absdiff(prev_gray, gray)
            score = float(np.count_nonzero(diff > 25)) / diff.size
            if score > best_score:
                best_score  = score
                best_frame  = frame.copy()
                best_sec    = i / fps

        prev_gray = gray

    cap.release()

    if best_frame is None:
        print("  No candidate frame found")
        return None

    print(f"  Peak motion: {best_score:.3f} at t={best_sec:.1f}s")
    print(f"  Running triage pipeline...")

    report = generate_triage_report(best_frame)
    report['_source_video'] = video_path.name
    report['_trigger_time_sec'] = round(best_sec, 2)

    t = report['_timing']
    print(f"  ✅ YOLO: {t['yolo_ms']}ms | VLM: {t['vlm_sec']}s | Total: {t['total_sec']}s")

    visualize_triage(
        best_frame, report,
        title=f"{video_path.name} — trigger at {best_sec:.1f}s"
    )

    return report


# ── Run on first video ─────────────────────────────────────
test_vids = sorted(CADP_VIDEOS.glob('*.mp4')) if CADP_VIDEOS.exists() else []

if test_vids:
    video_report = run_triage_on_video(test_vids[0])

    if video_report:
        # Save
        out_file = RESULTS_DIR / f"video_triage_{test_vids[0].stem}.json"
        save_data = {k: v for k, v in video_report.items()
                    if not k.startswith('_yolo') and k != '_raw_output'}
        with open(out_file, 'w') as f:
            json.dump(save_data, f, indent=2)
        print(f"\n💾 Report saved to {out_file}")
else:
    print("❌ No videos found in forth_investigation/")
    print("   You can still test with dataset frames in cell 4.1")

# ============================================================
# 4.3  Batch evaluation on multiple accident frames
#      Quality metrics: JSON validity, field completeness,
#      severity distribution, timing statistics
# ============================================================

from collections import Counter

acc_dir  = DATASET_DIR / 'test' / 'accident'
norm_dir = DATASET_DIR / 'test' / 'no_accident'

if not acc_dir.exists():
    print("❌ Run cell 4.1 first to verify the dataset path")
else:
    N_ACC  = 8   # accident frames to test
    N_NORM = 4   # normal frames — should get low severity / no alert

    acc_frames  = random.sample(list(acc_dir.glob('*.jpg')),
                                min(N_ACC, len(list(acc_dir.glob('*.jpg')))))
    norm_frames = random.sample(list(norm_dir.glob('*.jpg')),
                                min(N_NORM, len(list(norm_dir.glob('*.jpg')))))

    all_frames = [(p, 'accident') for p in acc_frames] +                  [(p, 'normal')   for p in norm_frames]

    results = []
    print(f"Running triage on {len(all_frames)} frames ({N_ACC} accident, {N_NORM} normal)\n")

    for i, (path, gt_label) in enumerate(all_frames):
        frame_bgr = cv2.imread(str(path))
        if frame_bgr is None:
            continue

        report = generate_triage_report(frame_bgr)
        t = report['_timing']

        severity = report.get('severity_score', 0)
        alert    = report.get('alert_level', 'UNKNOWN')
        valid_json = '_parse_error' not in report

        results.append({
            'gt': gt_label, 'severity': severity,
            'alert': alert, 'valid_json': valid_json,
            'vlm_sec': t['vlm_sec'], 'total_sec': t['total_sec'],
        })

        icon = '🚨' if gt_label == 'accident' else '🟢'
        print(f"{icon} [{i+1:2d}/{len(all_frames)}] {gt_label:8s} | "
              f"alert={alert:8s} sev={severity} | "
              f"t={t['total_sec']}s | json={'✅' if valid_json else '❌'}")

    # ── Summary stats ─────────────────────────────────────
    print("\n" + "="*60)
    print("📊 BATCH EVALUATION SUMMARY")
    print("="*60)

    acc_res  = [r for r in results if r['gt'] == 'accident']
    norm_res = [r for r in results if r['gt'] == 'normal']

    print(f"\nAccident frames ({len(acc_res)}):")
    print(f"  Avg severity : {sum(r['severity'] for r in acc_res)/max(len(acc_res),1):.1f}/5")
    print(f"  Alert dist   : {dict(Counter(r['alert'] for r in acc_res))}")
    print(f"  HIGH+CRITICAL: {sum(1 for r in acc_res if r['alert'] in ('HIGH','CRITICAL'))}/{len(acc_res)}")

    print(f"\nNormal frames ({len(norm_res)}):")
    print(f"  Avg severity : {sum(r['severity'] for r in norm_res)/max(len(norm_res),1):.1f}/5")
    print(f"  Alert dist   : {dict(Counter(r['alert'] for r in norm_res))}")

    all_times = [r['vlm_sec'] for r in results]
    print(f"\nTiming (VLM only):")
    print(f"  Min: {min(all_times):.2f}s | Max: {max(all_times):.2f}s | "
          f"Avg: {sum(all_times)/len(all_times):.2f}s")
    print(f"  Successful JSON: {sum(r['valid_json'] for r in results)}/{len(results)}")
    print("="*60)

    # Save batch results
    out_file = RESULTS_DIR / 'batch_evaluation.json'
    with open(out_file, 'w') as f:
        json.dump(results, f, indent=2)
    print(f"\n💾 Batch results saved to {out_file}")

In [ ]:
# ============================================================
# 4.4  Print a formatted sample triage report
#      (shows exactly what a dispatcher would receive)
# ============================================================

def print_dispatcher_alert(report: dict, video_name: str = "CAM-01"):
    """Format a triage report as a dispatcher-style alert."""
    alert   = report.get('alert_level', 'UNKNOWN')
    sev     = report.get('severity_score', '?')
    color   = {'LOW': '\033[92m', 'MEDIUM': '\033[93m',
                'HIGH': '\033[91m', 'CRITICAL': '\033[95m'}.get(alert, '')
    reset   = '\033[0m'

    units = ', '.join(report.get('recommended_units', [])) or 'TBD'
    vehs  = ', '.join(report.get('vehicles_involved', [])) or 'unknown'
    haz   = ', '.join(report.get('hazards', [])) or 'none reported'

    print(f"{color}{'='*55}")
    print(f"  ⚠  ACCIDENT ALERT — {alert}  (Severity {sev}/5)")
    print(f"{'='*55}{reset}")
    print(f"  Camera    : {video_name}")
    print(f"  Time      : {time.strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"  Road type : {report.get('road_type', 'unknown')}")
    print(f"  Zone      : {report.get('collision_zone', 'unknown')}")
    print()
    print(f"  Scene: {report.get('scene_description', 'N/A')}")
    print()
    print(f"  Vehicles  : {vehs}  ({report.get('vehicle_count', '?')} total)")
    print(f"  Pedestrians detected: {report.get('pedestrians_in_scene', '?')}")
    print(f"  Hazards   : {haz}")
    print(f"  Visibility: {report.get('visibility_conditions', 'unknown')}")
    print()
    print(f"  ➤  Dispatch: {units}")
    t = report.get('_timing', {})
    print(f"  ➤  AI confidence: {report.get('confidence', '?')}")
    print(f"  ➤  Detection time: {t.get('total_sec', '?')}s")
    print(f"{'='*55}")

# ── Test with the most recent report ──────────────────────
report_files = sorted(RESULTS_DIR.glob('*.json'))
if report_files:
    with open(report_files[-1]) as f:
        saved = json.load(f)

    if isinstance(saved, list) and saved:
        sample = saved[0].get('report', saved[0])
    else:
        sample = saved

    print_dispatcher_alert(sample, video_name="CAM-INTERSECTION-04")
else:
    print("Run cell 4.1 or 4.2 first to generate a report.")

## Section 7 — Unified End-to-End Pipeline

In [ ]:
# ============================================================
# End-to-End run_pipeline() — connects Path1 → Path2 → VLM
# ============================================================
# This function orchestrates the full emergency response pipeline:
#   1. Path 1 (TimeSformer sliding window) detects accident + returns bbox
#   2. Path 2 (YOLO11 + ByteTrack) confirms with physics-calibrated detector
#   3. VLM (Qwen2.5-VL) generates structured triage report
#   4. Dispatcher alert is formatted and returned
# ============================================================

def run_pipeline(
    video_path,
    path1_thresh=0.6,
    path2_max_sec=120,
    use_path1=True,
    use_path2=True,
    use_vlm=True,
):
    """
    Full emergency response pipeline.

    Parameters
    ----------
    video_path    : path to accident video
    path1_thresh  : TimeSformer detection threshold (default 0.6)
    path2_max_sec : max seconds for Path2 ByteTrack (default 120)
    use_path1     : run Path1 TimeSformer inference
    use_path2     : run Path2 YOLO11+ByteTrack
    use_vlm       : run Qwen2.5-VL triage

    Returns
    -------
    dict with keys: path1_result, path2_result, vlm_report, alert, timing
    """
    import time
    from pathlib import Path as _Path
    video_path = _Path(video_path)
    result = {
        'video':        video_path.name,
        'path1_result': None,
        'path2_result': None,
        'vlm_report':   None,
        'alert':        None,
        'timing':       {}
    }

    trigger_frame_bgr = None
    accident_bbox     = None

    # ── Path 1: TimeSformer ──────────────────────────────
    if use_path1:
        t0 = time.time()
        try:
            print(f"\n[Path 1] TimeSformer sliding-window inference...")
            tl1, ev1 = infer_video_with_bbox(video_path, thresh=path1_thresh)
            result['path1_result'] = {'timeline': tl1, 'first_event': ev1}
            if ev1:
                trigger_frame_bgr = ev1.get('trigger_frame')
                accident_bbox     = ev1.get('collision_bbox')
                print(f"  ✅ Path1 alert at {ev1['time_sec']:.1f}s  bbox={accident_bbox}")
            else:
                print("  ℹ️  Path1: no accident detected")
        except Exception as e:
            print(f"  ⚠️  Path1 error: {e}")
        result['timing']['path1_sec'] = round(time.time() - t0, 2)

    # ── Path 2: YOLO11 + ByteTrack ────────────────────────
    if use_path2:
        t0 = time.time()
        try:
            print(f"\n[Path 2] YOLO11+ByteTrack physics-calibrated...")
            events2, tl2, snap2 = process_video(
                video_path, max_sec=path2_max_sec, visualize=False)
            result['path2_result'] = {
                'events': events2, 'timeline': tl2, 'snap': snap2}
            if events2:
                ev2 = events2[0]
                print(f"  ✅ Path2 event: {ev2['collision_type']} at {ev2['time_sec']}s")
                if trigger_frame_bgr is None and snap2 is not None:
                    trigger_frame_bgr = snap2
                if accident_bbox is None:
                    accident_bbox = ev2.get('collision_bbox_padded',
                                            ev2.get('collision_bbox'))
            else:
                print("  ℹ️  Path2: no collision events detected")
        except Exception as e:
            print(f"  ⚠️  Path2 error: {e}")
        result['timing']['path2_sec'] = round(time.time() - t0, 2)

    # ── VLM: Qwen2.5-VL Triage ────────────────────────────
    if use_vlm and trigger_frame_bgr is not None:
        t0 = time.time()
        try:
            print(f"\n[VLM] Qwen2.5-VL triage analysis...")
            report = generate_triage_report(trigger_frame_bgr)
            result['vlm_report'] = report
            alert = report.get('alert_level', 'UNKNOWN')
            sev   = report.get('severity_score', '?')
            print(f"  ✅ Alert: {alert} | Severity: {sev}/5")
            result['alert'] = {
                'level':    alert,
                'severity': sev,
                'units':    report.get('recommended_units', []),
                'scene':    report.get('scene_description', ''),
                'hazards':  report.get('hazards', []),
            }
        except Exception as e:
            print(f"  ⚠️  VLM error: {e}")
        result['timing']['vlm_sec'] = round(time.time() - t0, 2)
    elif use_vlm:
        print("\n[VLM] Skipped — no trigger frame available from Path1/Path2")

    total_sec = sum(result['timing'].values())
    result['timing']['total_sec'] = round(total_sec, 2)
    print(f"\n✅ Pipeline complete in {total_sec:.1f}s")
    print(f"   Timings: {result['timing']}")
    if result['alert']:
        print(f"   ⚠️  ALERT: {result['alert']['level']} "
              f"| Dispatch: {result['alert']['units']}")
    return result


# ── Quick test on first video ──────────────────────────────
test_vids = sorted(VIDEOS_DIR.glob('*.mp4')) if VIDEOS_DIR.exists() else []
if test_vids:
    print(f"Found {len(test_vids)} videos. Running pipeline on first video...")
    pipeline_result = run_pipeline(
        test_vids[0],
        use_path1=False,  # Set True after Section 4 models are loaded
        use_path2=True,
        use_vlm=False,    # Set True after Section 6 model is loaded
    )
else:
    print("No videos found — set VIDEOS_DIR path in Cell 2")

In [ ]:
# ============================================================
# Latency Benchmark
# ============================================================
# Measures end-to-end latency of each pipeline stage.
# Requires all models loaded (Sections 2-6).
# ============================================================

import time

def benchmark_pipeline(video_path, n_runs=3):
    """
    Run the pipeline n_runs times and report latency statistics.
    Results give per-stage timings in seconds.
    """
    timings = []
    for i in range(n_runs):
        print(f"\nRun {i+1}/{n_runs}...")
        res = run_pipeline(
            video_path,
            use_path1=False,   # Enable after loading Section 4 model
            use_path2=True,
            use_vlm=False,     # Enable after loading Section 6 model
        )
        timings.append(res['timing'])

    print("\n" + "="*55)
    print("LATENCY BENCHMARK RESULTS")
    print("="*55)
    stages = list(timings[0].keys())
    for stage in stages:
        vals = [t.get(stage, 0) for t in timings]
        print(f"  {stage:20s}: "
              f"avg={sum(vals)/len(vals):.2f}s "
              f"min={min(vals):.2f}s "
              f"max={max(vals):.2f}s")
    print("="*55)
    return timings


test_vids = sorted(VIDEOS_DIR.glob('*.mp4')) if VIDEOS_DIR.exists() else []
if test_vids:
    bench_timings = benchmark_pipeline(test_vids[0], n_runs=2)
else:
    print("No test videos found")

In [ ]:
# ============================================================
# Final Results Summary Table
# ============================================================

summary_rows = [
    {"Model": "EfficientNet-B0 (Section 2)", "Type": "Frame classifier",
     "Latency": "~5 ms/frame", "Notes": "Baseline — binary accident/normal"},
    {"Model": "YOLOv11-cls (Section 3)", "Type": "Frame classifier",
     "Latency": "~8 ms/frame", "Notes": "Ultralytics cls variant"},
    {"Model": "TimeSformer (Path 1, Sec 4)", "Type": "Video ViT (8 frames)",
     "Latency": "~120 ms/clip", "Notes": "Sliding window, fp16 AMP"},
    {"Model": "YOLO11+ByteTrack (Path 2, Sec 5)", "Type": "Tracking+physics",
     "Latency": "~25 ms/frame", "Notes": "BH/s² scale-invariant thresholds"},
    {"Model": "Qwen2.5-VL-7B (Section 6)", "Type": "VLM triage",
     "Latency": "~4-6 s/event", "Notes": "NF4 quantized, fires once per event"},
]

try:
    import pandas as pd
    df = pd.DataFrame(summary_rows)
    print("\n" + "="*70)
    print("EMERGENCY RESPONSE SYSTEM — MODEL SUMMARY")
    print("="*70)
    print(df.to_string(index=False))
    print("="*70)
except ImportError:
    print("pandas not installed — printing raw summary:")
    for r in summary_rows:
        print(f"  [{r['Model']}] {r['Type']} | {r['Latency']} | {r['Notes']}")

# ── Load and display saved triage results if available ────────
result_files = sorted(RESULTS_DIR.glob('*.json'))
if result_files:
    print(f"\nSaved triage results ({len(result_files)} files):")
    for rf in result_files:
        print(f"  📄 {rf.name}")
else:
    print("\nNo triage result files found yet — run Sections 2-6 first")

print("\n✅ GP Emergency Response System — pipeline ready.")
print("   Run sections in order: 1 → 2 → 3 → 4 → 5 → 6 → 7")